# Авито: весь код и обучение в одном ноутбуке

Здесь заново готовятся признаки, обучаются две модели CatBoost и создаётся `answer.csv`.
Готовых моделей конкурса и таблиц признаков в загружаемом комплекте нет.
Открытая языковая модель E5 используется как предобученный кодировщик, её веса не обучаются заново.

В Colab выберите **Runtime → Change runtime type → GPU**, затем выполняйте ячейки сверху вниз.
Python 3.12 для обучения устанавливается автоматически в отдельное окружение. Версию Python ядра Colab вручную менять не нужно [7].
Планируйте около 16 ГБ RAM и 20 ГБ диска с запасом. Полный запуск в Colab ещё не измерен.
Ресурсы и время сессии Colab ограничены [2].

После полного переобучения на CUDA новый CSV может отличаться от прежнего ответа с Apple MPS.
В конце сохраняются метрика этого запуска и контрольная сумма его CSV.

Весь код решения находится в ячейках этого файла. Ячейки `%%writefile` сохраняют показанный код в рабочую папку. Каждый этап затем выполняется отдельным процессом и освобождает свою память после завершения. Загружаются только библиотеки, исходные данные и открытые веса E5.

Время обучения и объём оперативной памяти остаются примерно такими же, как у варианта с внешними исходниками. Само сохранение текста кода почти не влияет на ресурсы.

## 1. Рабочая папка и параметры

Для первого полного запуска используется новая папка `avito_standalone`. Все параметры обучения показаны в этой ячейке. Базовые версии библиотек закреплены.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

NOTEBOOK_START_DIR = globals().get("NOTEBOOK_START_DIR", Path.cwd())
IN_COLAB = Path("/content").is_dir()
WORK = (
    Path("/content/avito_standalone")
    if IN_COLAB
    else NOTEBOOK_START_DIR / "run_standalone"
)
DEVICE = "cuda" if IN_COLAB else "auto"
DATASET_ARCHIVE = None

WORK.mkdir(parents=True, exist_ok=True)
for name in [
    "scripts",
    "src/avito_retrieval",
    "configs",
    "reports",
    "artifacts",
    "data",
]:
    (WORK / name).mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
(WORK / "src/avito_retrieval/__init__.py").write_text("")

REQUIREMENTS = [
    "numpy==1.26.4",
    "pandas==2.3.3",
    "pyarrow==22.0.0",
    "scipy==1.13.1",
    "scikit-learn==1.8.0",
    "catboost==1.2.8",
    "nltk==3.9.1",
    "transformers==5.6.2",
    "huggingface-hub==1.12.0",
]
(WORK / "requirements.txt").write_text("\n".join(REQUIREMENTS) + "\n")

CONFIG = {
    "split_seed": 20260928,
    "validation_count": 6000,
    "dense": {
        "batch_size": 32,
        "checkpoint_rows": 512,
        "item_max_length": 256,
        "query_max_length": 128,
        "precision": "float32",
        "seed": 42,
    },
    "validation_features": "artifacts/features_hybrid.parquet",
    "benchmark_features": "artifacts/features_benchmark.parquet",
    "queries_ids": "data/ids/benchmark_queries.parquet",
    "allowed_items_ids": "data/ids/benchmark_items.parquet",
    "corpus_ids": "data/ids/corpus.parquet",
    "models": [
        {
            "path": "artifacts/final_d6.cbm",
            "loss": "Logloss",
            "iterations": 700,
            "depth": 6,
            "learning_rate": 0.07,
            "seed": 42,
            "threads": 4,
            "drop_features": ["category_match"],
        },
        {
            "path": "artifacts/final_no_history.cbm",
            "loss": "Logloss",
            "iterations": 1000,
            "depth": 6,
            "learning_rate": 0.07,
            "seed": 42,
            "threads": 4,
            "drop_features": [
                "category_match",
                "popularity",
                "memory_score",
                "text_item_count",
                "context_item_count",
                "rank_memory",
                "rank_geo_memory",
            ],
        },
    ],
    "weights": [0.5, 0.5],
    "reference_answer_sha256": "3ebda47a966aa2920fe9480cb3b4b1602ec91b63f207d89775d981d16995e3c6",
}
(WORK / "configs/final.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
print("Рабочая папка:", WORK)
print("Свободный диск, ГБ:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Установить библиотеки

Библиотеки устанавливаются в отдельное окружение. Версии numpy в основном ядре Colab
не влияют на дочерние процессы обучения, перезапуск ядра после установки не нужен.
PyTorch загружается из официального репозитория [3].
Отдельный Python 3.12 загружается с помощью uv [7]. Все этапы используют именно этот интерпретатор, независимо от версии ядра Colab.

In [ ]:
# Python для обучения устанавливается отдельно от Python ядра Colab.
bootstrap = WORK / ".uv-bootstrap"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--target",
        str(bootstrap),
        "uv==0.10.11",
    ],
    check=True,
)
uv_env = {
    **os.environ,
    "PYTHONPATH": str(bootstrap),
    "UV_CACHE_DIR": str(WORK / ".uv-cache"),
    "UV_PYTHON_INSTALL_DIR": str(WORK / ".python"),
    "UV_PYTHON_BIN_DIR": str(WORK / ".python-bin"),
}
venv = WORK / ".venv312"
PYTHON = venv / "bin/python"
version_probe = (
    "import sys" + chr(10) + "print('.'.join(map(str, sys.version_info[:2])))"
)
existing_version = None
if PYTHON.exists():
    check = subprocess.run(
        [str(PYTHON), "-c", version_probe], capture_output=True, text=True
    )
    if check.returncode == 0:
        existing_version = check.stdout.strip()
if existing_version != "3.12":
    if venv.exists():
        # Сохраняем прежнее окружение, если оно создано другой версией Python.
        import time

        venv.rename(WORK / (".venv312-backup-" + str(time.time_ns())))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "uv",
            "venv",
            "--python",
            "3.12",
            "--managed-python",
            "--seed",
            str(venv),
        ],
        env=uv_env,
        check=True,
    )
actual_python = subprocess.check_output(
    [str(PYTHON), "-c", version_probe], text=True
).strip()
assert actual_python == "3.12", actual_python
subprocess.run([str(PYTHON), "-m", "pip", "--version"], check=True)
print("Python ядра:", sys.version.split()[0])
print("Python обучения:", actual_python)
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(WORK / "requirements.txt"),
    ],
    check=True,
)
torch_command = [str(PYTHON), "-m", "pip", "install", "--quiet", "torch==2.9.1"]
if sys.platform.startswith("linux"):
    wheel_kind = "cu126" if DEVICE in ["auto", "cuda"] else "cpu"
    torch_command += ["--index-url", "https://download.pytorch.org/whl/" + wheel_kind]
subprocess.run(torch_command, check=True)
probe = (
    "import json, torch, platform"
    + chr(10)
    + "print(json.dumps({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
)
environment = json.loads(subprocess.check_output([str(PYTHON), "-c", probe], text=True))
print(environment)
if DEVICE == "cuda" and not environment["cuda"]:
    raise RuntimeError(
        "GPU недоступен. Включите GPU в настройках Colab или явно выберите DEVICE = cpu."
    )
(WORK / "reports/environment.json").write_text(json.dumps(environment, indent=2))


def run_stage(stage):
    command = [
        str(PYTHON),
        "-u",
        str(WORK / "scripts/from_scratch.py"),
        "--stage",
        stage,
        "--device",
        DEVICE,
    ]
    if DATASET_ARCHIVE and stage == "download":
        command += ["--dataset-archive", str(DATASET_ARCHIVE)]
    subprocess.run(command, cwd=WORK, check=True)


## 3. Управление этапами

Код ниже последовательно вызывает подготовку данных, построение признаков, обучение и проверку. Каждая команда и время её выполнения записываются в `reports/run_steps.json`. Все вызываемые файлы создаются следующими ячейками этого же ноутбука.

In [ ]:
%%writefile scripts/from_scratch.py
"""Запустить один этап полного обучения, не подгружая готовые модели конкурса."""

import argparse
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import time

ROOT = Path(__file__).resolve().parents[1]
STAGES = [
    "download",
    "split",
    "lexical",
    "model",
    "embeddings",
    "train_features",
    "train",
    "audit",
    "benchmark",
    "predict",
]


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def commands(stage, config, device, archive=None):
    dense = config["dense"]
    model_args = []
    for model in config["models"]:
        model_args.extend(["--model", model["path"]])
    if stage == "download":
        return [("download_data.py", *(["--archive", str(archive)] if archive else []))]
    if stage == "split":
        return [
            (
                "prepare_validation.py",
                "--seed",
                str(config["split_seed"]),
                "--queries",
                str(config["validation_count"]),
            )
        ]
    if stage == "lexical":
        return [
            (
                "prepare_lexical.py",
                "--items",
                "data/processed/corpus.parquet",
                "--output",
                "artifacts/lexical.joblib",
            )
        ]
    if stage == "model":
        return [("prepare_dense.py", "--download-model", "--download-only")]
    if stage == "embeddings":
        common = [
            "--device",
            device,
            "--batch-size",
            str(dense["batch_size"]),
            "--checkpoint-rows",
            str(dense["checkpoint_rows"]),
            "--max-length",
            str(dense["item_max_length"]),
            "--query-max-length",
            str(dense["query_max_length"]),
            "--precision",
            dense["precision"],
            "--seed",
            str(dense["seed"]),
        ]
        result = [
            (
                "prepare_dense.py",
                "--kind",
                "items",
                "--items-file",
                "data/processed/corpus.parquet",
                *common,
            )
        ]
        for name, source in [
            ("eval_queries", "data/processed/eval_queries.parquet"),
            ("benchmark_queries", "data/raw/benchmark_queries.parquet"),
        ]:
            for plain in [False, True]:
                result.append(
                    (
                        "prepare_dense.py",
                        "--kind",
                        "queries",
                        "--queries-file",
                        source,
                        "--query-name",
                        name + ("_plain" if plain else ""),
                        *(["--query-text-only"] if plain else []),
                        *common,
                    )
                )
        return result
    if stage in ["train_features", "benchmark"]:
        return [
            (
                "build_features.py",
                "--mode",
                "validation" if stage == "train_features" else "benchmark",
                "--dense",
                "--plain-query",
                "--output",
                (
                    config["validation_features"]
                    if stage == "train_features"
                    else config["benchmark_features"]
                ),
            )
        ]
    if stage == "train":
        result = []
        for model in config["models"]:
            args = [
                "train_ranker.py",
                "--features",
                config["validation_features"],
                "--output",
                model["path"],
                "--fit-splits",
                "fit,dev",
                "--fixed-iterations",
            ]
            for key in [
                "loss",
                "iterations",
                "depth",
                "learning_rate",
                "seed",
                "threads",
            ]:
                args.extend(["--" + key.replace("_", "-"), str(model[key])])
            args.extend(["--drop-features", ",".join(model["drop_features"])])
            result.append(tuple(args))
        return result
    if stage == "audit":
        return [
            (
                "compare_models.py",
                "--features",
                config["validation_features"],
                *model_args,
                "--split",
                "audit",
                "--blend-grid",
                "0.5",
                "--output",
                "reports/audit.json",
            )
        ]
    if stage == "predict":
        return [
            (
                "predict.py",
                "--features",
                config["benchmark_features"],
                *model_args,
                "--weights",
                *map(str, config["weights"]),
                "--queries",
                "data/raw/benchmark_queries.parquet",
                "--corpus",
                "data/processed/corpus.parquet",
                "--allowed-items",
                "data/raw/benchmark_items.parquet",
                "--output",
                "answer.csv",
            )
        ]
    raise ValueError(stage)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stage", choices=STAGES + ["all"], required=True)
    parser.add_argument(
        "--device", choices=["auto", "cuda", "cpu", "mps"], default="auto"
    )
    parser.add_argument("--config", type=Path, default=ROOT / "configs/final.json")
    parser.add_argument("--dataset-archive", type=Path)
    args = parser.parse_args()
    config_path = args.config.resolve()
    archive = args.dataset_archive.resolve() if args.dataset_archive else None
    config = json.loads(config_path.read_text())
    os.chdir(ROOT)
    os.environ.update(
        PYTHONPATH=str(ROOT / "src"),
        PYTHONHASHSEED="42",
        CUBLAS_WORKSPACE_CONFIG=":4096:8",
        OMP_NUM_THREADS="4",
        OPENBLAS_NUM_THREADS="3",
        TOKENIZERS_PARALLELISM="false",
    )
    for name in ["artifacts", "reports", "data"]:
        (ROOT / name).mkdir(exist_ok=True)
    path = ROOT / "reports/run_steps.json"
    history = json.loads(path.read_text()) if path.exists() else []
    for stage in STAGES if args.stage == "all" else [args.stage]:
        record = {
            "stage": stage,
            "started_at_unix": time.time(),
            "config_sha256": sha256(config_path),
            "commands": [],
            "status": "running",
        }
        history.append(record)
        path.write_text(json.dumps(history, indent=2) + "\n")
        try:
            with (ROOT / "reports" / f"{stage}.log").open("w") as log:
                for script, *arguments in commands(stage, config, args.device, archive):
                    command = [
                        sys.executable,
                        "-u",
                        str(ROOT / "scripts" / script),
                        *arguments,
                    ]
                    record["commands"].append([script, *arguments])
                    print("Запуск:", script, " ".join(arguments), flush=True)
                    process = subprocess.Popen(
                        command,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True,
                    )
                    for line in process.stdout:
                        print(line, end="", flush=True)
                        log.write(line)
                    if process.wait():
                        raise RuntimeError(
                            f"Этап {stage} завершился с ошибкой. См. reports/{stage}.log"
                        )
            record["status"] = "passed"
        except BaseException:
            record["status"] = "failed"
            raise
        finally:
            record["elapsed_seconds"] = round(
                time.time() - record["started_at_unix"], 3
            )
            path.write_text(json.dumps(history, indent=2) + "\n")
    if args.stage in ["all", "predict"]:
        actual = sha256(ROOT / "answer.csv")
        result = {
            "answer_sha256": actual,
            "reference_answer_sha256": config["reference_answer_sha256"],
            "matches_previous_mps_answer": actual == config["reference_answer_sha256"],
            "python": sys.version,
            "platform": platform.platform(),
            "config": config,
            "packages": {
                name: importlib.metadata.version(name)
                for name in [
                    "numpy",
                    "pandas",
                    "pyarrow",
                    "scikit-learn",
                    "catboost",
                    "torch",
                    "transformers",
                    "huggingface-hub",
                ]
            },
            "source_sha256": {
                str(p.relative_to(ROOT)): sha256(p)
                for folder in ["scripts", "src"]
                for p in sorted((ROOT / folder).rglob("*.py"))
            },
        }
        (ROOT / "reports/reproducibility.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2) + "\n"
        )
        print("SHA256 нового ответа:", actual)
        print(
            "Совпадение с прежним ответом MPS:", result["matches_previous_mps_answer"]
        )


if __name__ == "__main__":
    main()


## 4. Скачать исходные данные

Здесь полностью показаны загрузка архива, проверка SHA256, безопасная распаковка и проверка числа строк. Если Яндекс Диск недоступен, загрузите `dataset.zip` вручную и укажите путь в `DATASET_ARCHIVE`.

In [ ]:
%%writefile scripts/download_data.py
"""Получить исходный архив конкурса или проверить его локальную копию."""

import argparse
import hashlib
import json
import os
from pathlib import Path
from pathlib import PurePosixPath
import shutil
import stat
import tempfile
from urllib.parse import urlencode
from urllib.parse import urlparse
from urllib.request import urlopen
import zipfile


ROOT = Path(__file__).resolve().parents[1]
PUBLIC_KEY = "https://disk.yandex.ru/d/sNhfo0YOjGtufg"
DOWNLOAD_API = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
ARCHIVE_SHA256 = "8dd3cba59201bae333c11db89c5111198fa10cc52a70c70bdc57bd6a248fb777"
EXPECTED_ROWS = {
    "train.parquet": 497673,
    "benchmark_queries.parquet": 2452,
    "benchmark_items.parquet": 189212,
}
CHUNK_SIZE = 4 * 1024 * 1024


def check_archive_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    actual = digest.hexdigest()
    if actual != ARCHIVE_SHA256:
        raise ValueError(f"SHA256 архива не совпал: {actual}")
    return actual


def save_archive(stream, destination):
    temporary = destination.with_suffix(destination.suffix + ".part")
    try:
        with temporary.open("wb") as output:
            shutil.copyfileobj(stream, output, length=CHUNK_SIZE)
        check_archive_hash(temporary)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)


def prepare_archive(data_dir, source=None):
    destination = data_dir / "dataset.zip"
    if source is not None:
        source = source.expanduser().resolve()
        if source == destination.resolve():
            check_archive_hash(destination)
        else:
            with source.open("rb") as stream:
                save_archive(stream, destination)
        return destination

    if destination.exists():
        check_archive_hash(destination)
        return destination

    # Сеть используется только при явном запуске этого скрипта без готового архива.
    api_url = DOWNLOAD_API + "?" + urlencode({"public_key": PUBLIC_KEY})
    with urlopen(api_url, timeout=60) as response:
        href = json.load(response)["href"]
    if urlparse(href).scheme != "https":
        raise ValueError("Сервис загрузки вернул ссылку без HTTPS")
    with urlopen(href, timeout=120) as stream:
        save_archive(stream, destination)
    return destination


def archive_members(archive):
    members = {}
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in info.filename:
            raise ValueError(f"Недопустимый путь в архиве: {info.filename!r}")
        if info.is_dir():
            continue
        mode = info.external_attr >> 16
        if stat.S_ISLNK(mode) or stat.S_IFMT(mode) not in (0, stat.S_IFREG):
            raise ValueError(f"Недопустимый тип файла: {info.filename!r}")
        if path.name not in EXPECTED_ROWS or path.name in members:
            raise ValueError(f"Лишний или повторный файл в архиве: {info.filename!r}")
        members[path.name] = info
    if set(members) != set(EXPECTED_ROWS):
        raise ValueError("В архиве должны быть все три исходных Parquet-файла")
    return members


def verify_row_counts(directory):
    try:
        import pyarrow.parquet as parquet
    except ModuleNotFoundError as error:
        if error.name != "pyarrow":
            raise
        return {"checked": False, "reason": "pyarrow не установлен"}

    actual = {}
    for name, expected in EXPECTED_ROWS.items():
        rows = parquet.ParquetFile(directory / name).metadata.num_rows
        if rows != expected:
            raise ValueError(
                f"Неверное число строк в {name}: {rows}, ожидалось {expected}"
            )
        actual[name] = rows
    return {"checked": True, "rows": actual}


def extract_data(archive_path, data_dir):
    raw_dir = data_dir / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    # Сначала проверяем всю распаковку, затем заменяем готовые файлы по одному.
    with tempfile.TemporaryDirectory(prefix=".extract-", dir=data_dir) as temporary:
        staging = Path(temporary)
        with zipfile.ZipFile(archive_path) as archive:
            for name, member in archive_members(archive).items():
                with (
                    archive.open(member) as source,
                    (staging / name).open("wb") as target,
                ):
                    shutil.copyfileobj(source, target, length=CHUNK_SIZE)
        verification = verify_row_counts(staging)
        for name in EXPECTED_ROWS:
            os.replace(staging / name, raw_dir / name)
    return verification


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--archive", type=Path, help="Готовый dataset.zip, без обращения к сети"
    )
    parser.add_argument("--data-dir", type=Path, default=ROOT / "data")
    args = parser.parse_args()
    args.data_dir.mkdir(parents=True, exist_ok=True)
    archive = prepare_archive(args.data_dir, args.archive)
    verification = extract_data(archive, args.data_dir)
    print(
        json.dumps(
            {
                "archive": str(archive),
                "archive_sha256": ARCHIVE_SHA256,
                "raw_directory": str(args.data_dir / "raw"),
                "parquet": verification,
            },
            ensure_ascii=False,
            indent=2,
        )
    )


if __name__ == "__main__":
    main()


In [ ]:
run_stage("download")


## 5. Нормализация текста и разделение данных

Обработка русского текста, ключи контекста и создание fit/dev/audit показаны ниже. Проверяемые пары удаляются из обучающей истории. Настройки моделей зафиксированы заранее, audit не используется для подбора.

In [ ]:
%%writefile src/avito_retrieval/text.py
"""Одинаковая обработка текстов при обучении и поиске."""

import re
import unicodedata
from functools import lru_cache

from nltk.stem.snowball import RussianStemmer


TOKEN = re.compile(r"[a-zа-я0-9]+")
STEMMER = RussianStemmer()


def normalize(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    return " ".join(TOKEN.findall(text))


@lru_cache(maxsize=200_000)
def stem(token):
    return STEMMER.stem(token)


def tokenize(text):
    # Числа и латиница нужны для марок, моделей и размеров техники.
    return [stem(token) for token in normalize(text).split()]


def query_keys(frame):
    columns = [
        "search_query",
        "search_location_id",
        "search_is_delivery_search",
        "search_infm_params_text",
        "search_category",
    ]
    values = frame[columns].fillna("").astype(str)
    values["search_query"] = values["search_query"].map(normalize)
    values["search_infm_params_text"] = values["search_infm_params_text"].map(normalize)
    return values.agg("\x1f".join, axis=1)


In [ ]:
%%writefile scripts/prepare_validation.py
"""Готовим отложенные запросы с новыми текстами и объявлениями."""

import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from avito_retrieval.text import normalize, query_keys


SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
META_COLUMNS = ["item_id", "item_location_id", "item_category_id", "item_microcat_id"]


def make_validation(data_dir, output_dir, count=6000, seed=20260928):
    rng = np.random.default_rng(seed)
    train = pd.read_parquet(
        data_dir / "train.parquet", columns=SEARCH_COLUMNS + META_COLUMNS
    )
    train["text_key"] = train["search_query"].map(normalize)
    train["context_key"] = query_keys(train)
    pairs = train.drop_duplicates(["context_key", "item_id"])
    contexts = pairs.groupby("context_key", sort=True).agg(
        text_key=("text_key", "first"),
        positives=("item_id", list),
    )
    text_contexts = contexts.groupby("text_key", sort=True).size()
    selected = []
    used_texts = set()

    # Доли отражают наблюдаемую новизну benchmark, но не используют его ответы.
    for mode, size in [
        ("warm_context", round(count * 0.044)),
        ("warm_text", round(count * 0.33)),
        ("cold_text", count - round(count * 0.044) - round(count * 0.33)),
    ]:
        available = contexts[~contexts["text_key"].isin(used_texts)]
        if mode == "warm_context":
            available = available[available["positives"].map(len) >= 2]
        elif mode == "warm_text":
            available = available[available["text_key"].map(text_contexts) >= 2]
        by_text = available.groupby("text_key", sort=True).indices
        chosen_texts = rng.choice(sorted(by_text), size=size, replace=False)
        for text in chosen_texts:
            positions = by_text[text]
            row = available.iloc[int(rng.choice(positions))]
            targets = sorted(row["positives"])
            if mode == "warm_context":
                targets = sorted(
                    rng.choice(
                        targets, max(1, len(targets) // 2), replace=False
                    ).tolist()
                )
            selected.append(
                {
                    "context_key": row.name,
                    "text_key": text,
                    "regime": mode,
                    "targets": targets,
                }
            )
        used_texts.update(chosen_texts)

    selected = (
        pd.DataFrame(selected).iloc[rng.permutation(count)].reset_index(drop=True)
    )
    selected["split"] = np.repeat(
        ["fit", "dev", "audit"],
        [count // 2, count // 4, count - count // 2 - count // 4],
    )
    selected["query_id"] = selected["context_key"].map(
        lambda key: hashlib.sha256(key.encode()).hexdigest()[:16]
    )
    query_features = train.drop_duplicates("context_key").set_index("context_key")[
        SEARCH_COLUMNS
    ]
    queries = selected.drop(columns="targets").join(query_features, on="context_key")
    truth = (
        selected[["query_id", "split", "targets"]]
        .explode("targets")
        .rename(columns={"targets": "item_id"})
    )

    cold_texts = set(selected.loc[selected["regime"] == "cold_text", "text_key"])
    target_ids = sorted(set(truth["item_id"]))
    cold_items = set(
        rng.choice(target_ids, round(len(target_ids) * 0.904), replace=False)
    )
    hidden_pairs = set(
        zip(
            selected.loc[
                selected.index.repeat(selected["targets"].map(len)), "context_key"
            ],
            selected["targets"].explode(),
        )
    )
    forbidden_pair = pd.MultiIndex.from_frame(train[["context_key", "item_id"]]).isin(
        hidden_pairs
    )
    keep = (
        ~train["text_key"].isin(cold_texts)
        & ~train["item_id"].isin(cold_items)
        & ~forbidden_pair
    )
    retrieval_train = train.loc[keep].copy()

    # Корпус сохраняет тексты отложенных объявлений, но их выборы скрыты от обучения.
    # Негативы - весь реальный корпус, а не облегчённая случайная подвыборка.
    items = pd.read_parquet(data_dir / "benchmark_items.parquet")
    extra_ids = sorted(set(target_ids) - set(items["item_id"]))
    extra_items = pd.read_parquet(
        data_dir / "train.parquet", filters=[("item_id", "in", extra_ids)]
    )
    extra_items = extra_items[items.columns].drop_duplicates("item_id")
    corpus = pd.concat([items, extra_items], ignore_index=True)
    queries["seen_text"] = queries["text_key"].isin(retrieval_train["text_key"])
    queries["seen_context"] = queries["context_key"].isin(
        retrieval_train["context_key"]
    )
    truth["seen_item"] = truth["item_id"].isin(retrieval_train["item_id"])

    assert (
        not pd.MultiIndex.from_frame(retrieval_train[["context_key", "item_id"]])
        .isin(hidden_pairs)
        .any()
    )
    assert corpus["item_id"].is_unique
    assert truth["item_id"].isin(corpus["item_id"]).all()
    assert queries["query_id"].is_unique
    assert not set(retrieval_train["text_key"]) & cold_texts
    assert not set(retrieval_train["item_id"]) & cold_items

    output_dir.mkdir(parents=True, exist_ok=True)
    retrieval_train.to_parquet(output_dir / "retrieval_train.parquet", index=False)
    queries.to_parquet(output_dir / "eval_queries.parquet", index=False)
    truth.to_parquet(output_dir / "eval_truth.parquet", index=False)
    corpus.to_parquet(output_dir / "corpus.parquet", index=False)
    report = {
        "seed": seed,
        "queries": len(queries),
        "truth_pairs": len(truth),
        "retrieval_train_rows": len(retrieval_train),
        "benchmark_corpus_size": len(items),
        "validation_corpus_size": len(corpus),
        "extra_validation_items": len(extra_items),
        "seen_text_fraction": queries["seen_text"].mean(),
        "seen_context_fraction": queries["seen_context"].mean(),
        "seen_target_item_fraction": truth["seen_item"].mean(),
        "split_counts": queries["split"].value_counts().to_dict(),
        "regimes": queries.groupby(["split", "regime"]).size().to_dict(),
        "positives_per_query": truth.groupby("query_id").size().describe().to_dict(),
        "note": "Корпус расширен отложенными объявлениями. Более плотный корпус и неполная разметка ограничивают перенос локальной оценки на платформу.",
    }
    report["regimes"] = {
        "/".join(key): value for key, value in report["regimes"].items()
    }
    Path("reports/validation_split.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False) + "\n"
    )
    print(json.dumps(report, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type=Path, default=Path("data/raw"))
    parser.add_argument("--output-dir", type=Path, default=Path("data/processed"))
    parser.add_argument("--queries", type=int, default=6000)
    parser.add_argument("--seed", type=int, default=20260928)
    args = parser.parse_args()
    make_validation(args.data_dir, args.output_dir, args.queries, args.seed)


In [ ]:
run_stage("split")
print((WORK / "reports/validation_split.json").read_text())


## 6. Текстовые индексы

Словный и символьный TF-IDF, BM25, разрешение ничьих и поиск по корпусу. Сначала сохраняется определение индекса, затем код его построения.

In [ ]:
%%writefile src/avito_retrieval/lexical.py
"""Три независимых текстовых индекса и поиск без плотной матрицы всего корпуса."""

from __future__ import annotations

import hashlib
import logging
from numbers import Integral

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from avito_retrieval.text import normalize, tokenize


LOGGER = logging.getLogger(__name__)
TEXT_COLUMNS = ("item_title_raw", "item_infm_params_text", "item_description_raw")


def _text_column(frame, name):
    if name not in frame:
        return [""] * len(frame)
    return [value if isinstance(value, str) else "" for value in frame[name]]


def _locations(frame, name):
    if name not in frame:
        return None
    result = []
    for value in frame[name]:
        if pd.isna(value):
            result.append("")
        elif isinstance(value, (float, np.floating)) and value.is_integer():
            result.append(str(int(value)))
        else:
            result.append(str(value))
    return np.asarray(result)


def _topk(indices, scores, k):
    """Разрешать ничьи по позиции объявления, включая границу top-k."""
    if len(scores) > k:
        threshold = np.partition(scores, len(scores) - k)[len(scores) - k]
        above = np.flatnonzero(scores > threshold)
        tied = np.flatnonzero(scores == threshold)
        tied = tied[np.argsort(indices[tied], kind="stable")[: k - len(above)]]
        selected = np.concatenate((above, tied))
        indices, scores = indices[selected], scores[selected]
    order = np.lexsort((indices, -scores))
    return indices[order], scores[order]


class LexicalIndex:
    """Словный TF-IDF, BM25 описаний и символьный TF-IDF заголовков.

    Матрицы хранятся как CSR float32: признаки × объявления. Поиск возвращает
    позиции строк исходного корпуса, -1 и score=0 обозначают отсутствие кандидата.
    """

    def __init__(
        self, batch_size=16, char_max_features=250_000, char_min_df=2, k1=1.2, b=0.65
    ):
        if batch_size < 1:
            raise ValueError("batch_size must be positive")
        if k1 <= 0 or not 0 <= b <= 1:
            raise ValueError("BM25 requires k1 > 0 and 0 <= b <= 1")
        self.batch_size = batch_size
        self.char_max_features = char_max_features
        self.char_min_df = char_min_df
        self.k1, self.b = k1, b

    def fit(self, items: pd.DataFrame) -> "LexicalIndex":
        if "item_id" not in items or items.empty:
            raise ValueError("items must contain a nonempty item_id column")
        ids = items["item_id"].tolist()
        if any(not isinstance(value, str) for value in ids) or len(set(ids)) != len(
            ids
        ):
            raise ValueError("item_id must contain unique strings")
        self.item_ids_ = np.asarray(ids)
        self.item_locations_ = _locations(items, "item_location_id")
        titles, params, descriptions = (
            _text_column(items, name) for name in TEXT_COLUMNS
        )
        descriptions = [value[:3000] for value in descriptions]
        short_texts = [f"{title} {param}" for title, param in zip(titles, params)]

        fingerprint = hashlib.sha256()
        locations = (
            self.item_locations_
            if self.item_locations_ is not None
            else [""] * len(ids)
        )
        for row in zip(ids, titles, params, descriptions, locations):
            for value in row:
                encoded = value.encode("utf-8")
                fingerprint.update(len(encoded).to_bytes(8, "little"))
                fingerprint.update(encoded)

        self.vectorizers_, self.matrices_ = {}, {}
        word_options = dict(
            tokenizer=tokenize, token_pattern=None, lowercase=False, dtype=np.float32
        )
        LOGGER.info("Building word TF-IDF for %d items", len(items))
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), sublinear_tf=True, **word_options
        )
        self.matrices_["word"] = vectorizer.fit_transform(short_texts).T.tocsr()
        self.vectorizers_["word"] = vectorizer

        LOGGER.info("Building BM25")
        # Повторы усиливают заголовок и параметры относительно рекламного описания.
        full_texts = (
            f"{title} {title} {title} {param} {param} {description}"
            for title, param, description in zip(titles, params, descriptions)
        )
        vectorizer = CountVectorizer(**word_options)
        counts = vectorizer.fit_transform(full_texts)
        lengths = np.asarray(counts.sum(axis=1)).ravel()
        average_length = max(float(lengths.mean()), 1.0)
        df = np.bincount(counts.indices, minlength=counts.shape[1]).astype(np.float32)
        idf = np.log1p((len(items) - df + 0.5) / (df + 0.5)).astype(np.float32)
        norms = self.k1 * (1 - self.b + self.b * lengths / average_length)
        # Положительный IDF и насыщение TF по формуле Lucene BM25:
        # https://www.elastic.co/blog/found-similarity-in-elasticsearch
        for start in range(0, len(items), 4096):
            end = min(start + 4096, len(items))
            left, right = counts.indptr[start], counts.indptr[end]
            values = counts.data[left:right]
            row_norms = np.repeat(
                norms[start:end], np.diff(counts.indptr[start : end + 1])
            )
            values *= (self.k1 + 1) / (values + row_norms)
            values *= idf[counts.indices[left:right]]
        self.matrices_["bm25"] = counts.T.tocsr()
        self.vectorizers_["bm25"] = vectorizer
        del counts, descriptions

        LOGGER.info("Building character TF-IDF")
        vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            preprocessor=normalize,
            lowercase=False,
            max_features=self.char_max_features,
            min_df=self.char_min_df,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.matrices_["char"] = vectorizer.fit_transform(short_texts).T.tocsr()
        self.vectorizers_["char"] = vectorizer
        matrix_bytes = sum(
            matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
            for matrix in self.matrices_.values()
        )
        self.metadata_ = {
            "schema_version": 1,
            "n_items": len(items),
            "corpus_sha256": fingerprint.hexdigest(),
            "fingerprint_fields": ["item_id", *TEXT_COLUMNS, "item_location_id"],
            "description_char_limit": 3000,
            "k1": self.k1,
            "b": self.b,
            "char_min_df": self.char_min_df,
            "char_max_features": self.char_max_features,
            "matrix_bytes": matrix_bytes,
            "channels": {
                name: {"features": matrix.shape[0], "nonzero": matrix.nnz}
                for name, matrix in self.matrices_.items()
            },
        }
        LOGGER.info(
            "Index ready, sparse matrices occupy %.2f GiB", matrix_bytes / 2**30
        )
        return self

    def search(
        self, queries: pd.DataFrame, top_k=300, allowed_n_items=None, include_local=True
    ) -> dict[str, tuple[np.ndarray, np.ndarray]]:
        """Искать глобально и внутри локации, не обрезая локальный поиск global top-k.

        allowed_n_items ограничивает поиск префиксом корпуса. При локальной
        проверке префикс позволяет исключить добавленные обучающие объявления.
        """
        if not hasattr(self, "matrices_"):
            raise ValueError("fit the index before search")
        if isinstance(top_k, bool) or not isinstance(top_k, Integral) or top_k < 1:
            raise ValueError("top_k must be a positive integer")
        n_items = len(self.item_ids_)
        if allowed_n_items is not None:
            if (
                not isinstance(allowed_n_items, Integral)
                or not 0 <= allowed_n_items <= n_items
            ):
                raise ValueError("allowed_n_items must be a valid corpus prefix length")
            n_items = int(allowed_n_items)
        query_texts = [
            f"{text} {param}"
            for text, param in zip(
                _text_column(queries, "search_query"),
                _text_column(queries, "search_infm_params_text"),
            )
        ]
        query_locations = _locations(queries, "search_location_id")
        local = (
            include_local
            and self.item_locations_ is not None
            and query_locations is not None
        )
        results = {}
        for name, matrix in self.matrices_.items():
            channel_names = [name, f"local_{name}"] if local else [name]
            for channel_name in channel_names:
                results[channel_name] = (
                    np.full((len(queries), top_k), -1, dtype=np.int32),
                    np.zeros((len(queries), top_k), dtype=np.float32),
                )
            for start in range(0, len(queries), self.batch_size):
                text_batch = query_texts[start : start + self.batch_size]
                query_matrix = self.vectorizers_[name].transform(text_batch)
                if name == "bm25":
                    query_matrix.data.fill(1)
                # В памяти только ненулевые совпадения небольшой пачки запросов.
                scores = (query_matrix @ matrix).tocsr()
                for offset in range(len(text_batch)):
                    row = start + offset
                    left, right = scores.indptr[offset : offset + 2]
                    indices, values = (
                        scores.indices[left:right],
                        scores.data[left:right],
                    )
                    keep = (indices < n_items) & (values > 0)
                    indices, values = indices[keep], values[keep]
                    best_indices, best_scores = _topk(indices, values, top_k)
                    size = len(best_indices)
                    results[name][0][row, :size] = best_indices
                    results[name][1][row, :size] = best_scores
                    if local and query_locations[row]:
                        keep = self.item_locations_[indices] == query_locations[row]
                        best_indices, best_scores = _topk(
                            indices[keep], values[keep], top_k
                        )
                        size = len(best_indices)
                        results[f"local_{name}"][0][row, :size] = best_indices
                        results[f"local_{name}"][1][row, :size] = best_scores
        return results


In [ ]:
%%writefile scripts/prepare_lexical.py
"""Построить воспроизводимые текстовые индексы из локального корпуса."""

import argparse
import json
import logging
import os
from pathlib import Path
import tempfile
import time

import joblib
import numpy as np
import pandas as pd

from avito_retrieval.lexical import LexicalIndex


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--items", type=Path, default=Path("data/raw/benchmark_items.parquet")
    )
    parser.add_argument("--output", type=Path, default=Path("artifacts/lexical.joblib"))
    parser.add_argument("--batch-size", type=int, default=16)
    args = parser.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
    start = time.monotonic()
    columns = [
        "item_id",
        "item_title_raw",
        "item_infm_params_text",
        "item_description_raw",
        "item_location_id",
    ]
    items = pd.read_parquet(args.items, columns=columns)
    index = LexicalIndex(batch_size=args.batch_size).fit(items)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            dir=args.output.parent, suffix=".joblib", delete=False
        ) as handle:
            temporary_path = Path(handle.name)
        # Без сжатия индекс быстрее загружается и поддерживает mmap_mode="r".
        joblib.dump(index, temporary_path, compress=0)
        os.replace(temporary_path, args.output)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    np.save(
        args.output.with_suffix(".item_ids.npy"), index.item_ids_, allow_pickle=False
    )
    metadata = {
        **index.metadata_,
        "source": str(args.items),
        "build_seconds": round(time.monotonic() - start, 3),
    }
    args.output.with_suffix(".metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(metadata, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
run_stage("lexical")


## 7. Эмбеддинги E5

Полный код загрузки фиксированных открытых весов, токенизации, усреднения скрытых состояний и нормализации векторов. Векторы объявлений и запросов рассчитываются заново. E5 остаётся предобученной языковой моделью [6].

In [ ]:
%%writefile src/avito_retrieval/dense.py
"""Локальные эмбеддинги E5 с фиксированной моделью и продолжением расчёта.

Pooling и префиксы соответствуют официальной карточке модели:
https://huggingface.co/intfloat/multilingual-e5-small
"""

from __future__ import annotations

import hashlib
import json
import os
import random
import time
from pathlib import Path
from typing import Sequence

import numpy as np

MODEL_ID = "intfloat/multilingual-e5-small"
MODEL_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
MODEL_FILES = (
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "sentencepiece.bpe.model",
    "README.md",
)
ITEM_COLUMNS = (
    "item_id",
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
)
QUERY_COLUMNS = ("query_id", "search_query", "search_infm_params_text")


def file_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _save_json(path: Path, value: dict | list) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    os.replace(temporary, path)


def download_model(model_dir: str | Path) -> Path:
    """Единственный сетевой шаг, последующий инференс работает только локально."""
    from huggingface_hub import snapshot_download

    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(model_dir),
        allow_patterns=list(MODEL_FILES),
        max_workers=3,
    )
    missing = [name for name in MODEL_FILES if not (model_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Неполная загрузка модели: {missing}")
    _save_json(
        model_dir / "provenance.json",
        {
            "model_id": MODEL_ID,
            "revision": MODEL_REVISION,
            "model_card": f"https://huggingface.co/{MODEL_ID}/blob/{MODEL_REVISION}/README.md",
            "sha256": {name: file_sha256(model_dir / name) for name in MODEL_FILES},
        },
    )
    return model_dir


def _text(value: object) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return str(value).strip()


def query_text(row: dict, include_filters: bool = True) -> str:
    parts = [_text(row.get("search_query"))]
    if include_filters:
        parts.append(_text(row.get("search_infm_params_text")))
    return "\n".join(filter(None, parts))


def item_text(row: dict, description_chars: int = 2000) -> str:
    # Заголовок и параметры идут первыми: длинное описание обрезается токенизатором.
    return "\n".join(
        filter(
            None,
            (
                _text(row.get("item_title_raw")),
                _text(row.get("item_infm_params_text")),
                _text(row.get("item_description_raw"))[:description_chars],
            ),
        )
    )


class E5Encoder:
    def __init__(
        self,
        model_dir: str | Path,
        device: str = "auto",
        seed: int = 42,
        threads: int = 4,
        precision: str = "float32",
    ) -> None:
        import torch
        import transformers
        from transformers import AutoModel, AutoTokenizer

        if device not in {"auto", "cpu", "mps", "cuda"}:
            raise ValueError("device должен быть auto, cpu, mps или cuda")
        if precision not in {"float32", "float16"}:
            raise ValueError("precision должен быть float32 или float16")
        model_dir = Path(model_dir)
        provenance_path = model_dir / "provenance.json"
        if not provenance_path.is_file():
            raise FileNotFoundError(
                "Модель не подготовлена. Сначала выполните --download-model."
            )
        self.provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
        if (self.provenance.get("model_id"), self.provenance.get("revision")) != (
            MODEL_ID,
            MODEL_REVISION,
        ):
            raise ValueError(
                "Локальная модель не соответствует закреплённой ревизии E5"
            )
        for name in MODEL_FILES:
            if file_sha256(model_dir / name) != self.provenance["sha256"].get(name):
                raise ValueError(f"Контрольная сумма модели не совпала: {name}")
        self.device = device
        if device == "auto":
            if torch.cuda.is_available():
                self.device = "cuda"
            elif torch.backends.mps.is_available():
                self.device = "mps"
        if self.device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA недоступна. В Colab включите GPU в настройках среды."
            )
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.use_deterministic_algorithms(True)
        if self.device == "cuda":
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True
        if self.device == "auto":
            self.device = "cpu"
        if self.device == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError(
                "MPS недоступен в текущем процессе. Выберите --device cpu."
            )
        if self.device == "cpu" and precision == "float16":
            raise ValueError("Для CPU используйте float32")
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.set_num_threads(threads)
        self.seed = seed
        self.precision = precision
        self.versions = {
            "torch": torch.__version__,
            "transformers": transformers.__version__,
        }
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        dtype = torch.float16 if precision == "float16" else torch.float32
        self.model = (
            AutoModel.from_pretrained(
                model_dir,
                local_files_only=True,
                dtype=dtype,
                attn_implementation="eager",
            )
            .to(self.device)
            .eval()
        )
        self.dimension = int(self.model.config.hidden_size)

    def encode(
        self,
        texts: Sequence[str],
        kind: str = "query",
        max_length: int = 256,
        batch_size: int = 32,
    ) -> np.ndarray:
        import torch
        import torch.nn.functional as functional

        if kind not in {"query", "passage"}:
            raise ValueError("kind должен быть query или passage")
        if batch_size < 1 or not 1 <= max_length <= 512:
            raise ValueError("Нужны batch_size > 0 и max_length в пределах 1..512")
        embeddings = np.empty((len(texts), self.dimension), dtype=np.float32)
        with torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                batch = [
                    f"{kind}: {text}" for text in texts[start : start + batch_size]
                ]
                encoded = self.tokenizer(
                    batch,
                    max_length=max_length,
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                ).to(self.device)
                hidden = self.model(**encoded).last_hidden_state.float()
                mask = encoded["attention_mask"].unsqueeze(-1).bool()
                pooled = hidden.masked_fill(~mask, 0).sum(1) / mask.sum(1).clamp_min(1)
                normalized = functional.normalize(pooled, p=2, dim=1)
                embeddings[start : start + len(batch)] = normalized.cpu().numpy()
        if not np.isfinite(embeddings).all():
            raise FloatingPointError("Модель вернула нечисловые эмбеддинги")
        return embeddings


def prepare_embeddings(
    parquet_path: str | Path,
    output_dir: str | Path,
    encoder: E5Encoder,
    kind: str = "items",
    batch_size: int = 32,
    max_length: int = 256,
    limit: int | None = None,
    checkpoint_rows: int = 256,
    output_name: str | None = None,
    query_text_only: bool = False,
) -> dict:
    """Порядок строк сохраняется, после сбоя повторяются только незафиксированные строки."""
    import pyarrow.parquet as pq

    if kind not in {"items", "queries"}:
        raise ValueError("kind должен быть items или queries")
    if query_text_only and kind != "queries":
        raise ValueError("query_text_only применим только к запросам")
    output_name = output_name or kind
    if not output_name.replace("_", "").isalnum():
        raise ValueError("output_name должен состоять из букв, цифр и подчёркиваний")
    if checkpoint_rows < 1 or batch_size < 1 or (limit is not None and limit < 1):
        raise ValueError("Размеры порций и limit должны быть положительными")
    parquet_path, output_dir = Path(parquet_path), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    columns = ITEM_COLUMNS if kind == "items" else QUERY_COLUMNS
    source = pq.ParquetFile(parquet_path)
    missing = set(columns) - set(source.schema_arrow.names)
    if missing:
        raise ValueError(
            f"В {parquet_path.name} отсутствуют колонки: {sorted(missing)}"
        )
    ids = source.read(columns=[columns[0]]).column(0).to_pylist()
    if limit is not None:
        ids = ids[:limit]
    if not ids or any(not isinstance(value, str) for value in ids):
        raise ValueError("Нужен непустой набор строковых идентификаторов")
    if len(set(ids)) != len(ids):
        raise ValueError("Идентификаторы должны быть уникальными")
    ids_sha = hashlib.sha256(
        json.dumps(ids, separators=(",", ":")).encode()
    ).hexdigest()
    config = {
        "format_version": 1,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_weights_sha256": encoder.provenance["sha256"]["model.safetensors"],
        "source_name": parquet_path.name,
        "source_sha256": file_sha256(parquet_path),
        "ids_sha256": ids_sha,
        "rows": len(ids),
        "dimension": encoder.dimension,
        "dtype": "float32",
        "precision": encoder.precision,
        "device": encoder.device,
        "seed": encoder.seed,
        "versions": encoder.versions,
        "kind": kind,
        "batch_size": batch_size,
        "max_length": max_length,
        "checkpoint_rows": checkpoint_rows,
        "text_version": 1,
        "description_chars": 2000,
        "prefix": "passage: " if kind == "items" else "query: ",
        "pooling": "masked_mean_l2",
    }
    if query_text_only:
        config["query_text_only"] = True
    metadata_path = output_dir / f"{output_name}.meta.json"
    embedding_path = output_dir / f"{output_name}.npy"
    ids_path = output_dir / f"{output_name}.ids.json"
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata["config"] != config:
            raise ValueError(
                "Конфигурация или входные данные изменились. Выберите другую output-dir."
            )
        if json.loads(ids_path.read_text(encoding="utf-8")) != ids:
            raise ValueError("Порядок идентификаторов в кеше изменился")
        output = np.load(embedding_path, mmap_mode="r+")
        if output.shape != (len(ids), encoder.dimension) or output.dtype != np.float32:
            raise ValueError("Форма или тип кеша эмбеддингов не совпали")
        if metadata["complete"]:
            if file_sha256(embedding_path) != metadata["embeddings_sha256"]:
                raise ValueError("Контрольная сумма эмбеддингов не совпала")
            print(f"{kind}: готовый кеш, {len(ids):,} строк", flush=True)
            return metadata
    else:
        if embedding_path.exists() or ids_path.exists():
            raise FileExistsError(
                "Обнаружен неполный кеш без метаданных. Выберите другую output-dir."
            )
        output = np.lib.format.open_memmap(
            embedding_path,
            mode="w+",
            dtype=np.float32,
            shape=(len(ids), encoder.dimension),
        )
        _save_json(ids_path, ids)
        metadata = {
            "config": config,
            "completed_rows": 0,
            "complete": False,
            "elapsed_seconds": 0.0,
        }
        _save_json(metadata_path, metadata)
    offset = 0
    resumed_at = int(metadata["completed_rows"])
    if not 0 <= resumed_at <= len(ids):
        raise ValueError("Некорректная позиция продолжения расчёта")
    started = time.monotonic()
    previous_seconds = float(metadata["elapsed_seconds"])
    for records in source.iter_batches(
        batch_size=checkpoint_rows, columns=list(columns)
    ):
        end = min(offset + len(records), len(ids))
        if end <= resumed_at:
            offset += len(records)
            continue
        if offset >= len(ids):
            break
        begin = max(offset, resumed_at)
        rows = records.slice(begin - offset, end - begin).to_pylist()
        texts = [
            (
                item_text(row)
                if kind == "items"
                else query_text(row, include_filters=not query_text_only)
            )
            for row in rows
        ]
        output[begin:end] = encoder.encode(
            texts,
            kind="passage" if kind == "items" else "query",
            max_length=max_length,
            batch_size=batch_size,
        )
        # Сначала сохраняем числа, затем позицию: сбой не оставит «готовые» пустые строки.
        output.flush()
        elapsed = time.monotonic() - started
        metadata.update(completed_rows=end, elapsed_seconds=previous_seconds + elapsed)
        _save_json(metadata_path, metadata)
        speed = (end - resumed_at) / max(elapsed, 1e-6)
        eta = (len(ids) - end) / max(speed, 1e-6)
        print(
            f"{kind}: {end:,}/{len(ids):,}, {speed:.1f} строк/с, осталось {eta / 60:.1f} мин",
            flush=True,
        )
        offset += len(records)
    metadata.update(complete=True, embeddings_sha256=file_sha256(embedding_path))
    _save_json(metadata_path, metadata)
    return metadata


def load_embeddings(
    output_dir: str | Path,
    kind: str = "items",
    verify_hash: bool = True,
) -> tuple[np.ndarray, list[str]]:
    output_dir = Path(output_dir)
    metadata = json.loads(
        (output_dir / f"{kind}.meta.json").read_text(encoding="utf-8")
    )
    if not metadata["complete"]:
        raise ValueError("Расчёт эмбеддингов ещё не завершён")
    embedding_path = output_dir / f"{kind}.npy"
    if verify_hash and file_sha256(embedding_path) != metadata["embeddings_sha256"]:
        raise ValueError("Контрольная сумма эмбеддингов не совпала")
    values = np.load(embedding_path, mmap_mode="r")
    ids = json.loads((output_dir / f"{kind}.ids.json").read_text(encoding="utf-8"))
    ids_sha = hashlib.sha256(
        json.dumps(ids, separators=(",", ":")).encode()
    ).hexdigest()
    if ids_sha != metadata["config"]["ids_sha256"]:
        raise ValueError("Контрольная сумма идентификаторов не совпала")
    if (
        values.shape != (len(ids), metadata["config"]["dimension"])
        or values.dtype != np.float32
        or metadata["completed_rows"] != len(ids)
    ):
        raise ValueError("Идентификаторы и эмбеддинги не совпадают по размеру")
    return values, ids


In [ ]:
%%writefile scripts/prepare_dense.py
#!/usr/bin/env python3
"""Подготовка E5 и возобновляемый расчёт эмбеддингов корпуса и запросов."""

from __future__ import annotations

import argparse
import json
from pathlib import Path
import sys

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from avito_retrieval.dense import E5Encoder, download_model, prepare_embeddings


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=ROOT / "data/raw")
    parser.add_argument(
        "--items-file",
        type=Path,
        help="Корпус для локальной оценки, включая heldout объявления",
    )
    parser.add_argument(
        "--queries-file",
        type=Path,
        help="Отдельный Parquet с query_id и признаками запроса",
    )
    parser.add_argument(
        "--query-name",
        default="benchmark_queries",
        help="Имя выходных файлов запросов без расширения",
    )
    parser.add_argument("--output-dir", type=Path, default=ROOT / "artifacts/dense")
    parser.add_argument(
        "--query-text-only",
        action="store_true",
        help="Кодировать текст запроса без фильтров поиска",
    )
    parser.add_argument(
        "--model-dir",
        type=Path,
        default=ROOT / "artifacts/models/multilingual-e5-small",
    )
    parser.add_argument(
        "--device", choices=("auto", "cpu", "mps", "cuda"), default="auto"
    )
    parser.add_argument(
        "--precision", choices=("float32", "float16"), default="float32"
    )
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-length", type=int, default=256)
    parser.add_argument("--query-max-length", type=int, default=128)
    parser.add_argument("--checkpoint-rows", type=int, default=256)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--limit", type=int, help="Число первых строк для замера скорости"
    )
    parser.add_argument("--kind", choices=("items", "queries", "both"), default="both")
    parser.add_argument(
        "--download-model",
        action="store_true",
        help="Однократно загрузить открытую модель",
    )
    parser.add_argument("--download-only", action="store_true")
    args = parser.parse_args()
    if args.download_only and not args.download_model:
        parser.error("--download-only требует --download-model")
    if args.download_model:
        download_model(args.model_dir)
    if args.download_only:
        return
    encoder = E5Encoder(
        args.model_dir, args.device, args.seed, args.threads, args.precision
    )
    print(
        f"E5: устройство {encoder.device}, вычисления {encoder.precision}, размерность {encoder.dimension}",
        flush=True,
    )
    kinds = ("items", "queries") if args.kind == "both" else (args.kind,)
    for kind in kinds:
        source = args.items_file if kind == "items" else args.queries_file
        source = source or args.data_dir / f"benchmark_{kind}.parquet"
        result = prepare_embeddings(
            source,
            args.output_dir,
            encoder,
            kind=kind,
            batch_size=args.batch_size,
            max_length=args.max_length if kind == "items" else args.query_max_length,
            limit=args.limit,
            checkpoint_rows=args.checkpoint_rows,
            output_name="items" if kind == "items" else args.query_name,
            query_text_only=args.query_text_only and kind == "queries",
        )
        print(
            json.dumps(
                {
                    "kind": kind,
                    "complete": result["complete"],
                    "rows": result["completed_rows"],
                    "seconds": result["elapsed_seconds"],
                }
            ),
            flush=True,
        )


if __name__ == "__main__":
    main()


In [ ]:
run_stage("model")
run_stage("embeddings")


## 8. Поведенческие и географические признаки

Статистики строятся по обучающим взаимодействиям. При подготовке локальной проверки скрытые пары уже исключены. Затем текстовые, смысловые и поведенческие кандидаты объединяются в одну таблицу признаков.

In [ ]:
%%writefile src/avito_retrieval/behavior.py
"""Статистики выбора и перенос знаний с похожих обучающих запросов."""

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as normalize_matrix

from avito_retrieval.text import normalize, query_keys, tokenize


class BehaviorIndex:
    def fit(self, train, items):
        self.item_ids = items["item_id"].to_numpy()
        self.item_categories = items["item_microcat_id"].to_numpy()
        self.category_ids = np.sort(train["item_microcat_id"].unique())
        self.category_lookup = {value: i for i, value in enumerate(self.category_ids)}
        self.item_category_indices = np.array(
            [self.category_lookup.get(value, -1) for value in self.item_categories]
        )
        train = train.copy()
        train["text_key"] = train["search_query"].map(normalize)
        train["context_key"] = query_keys(train)
        self.texts = np.sort(train["text_key"].unique())
        text_lookup = {value: i for i, value in enumerate(self.texts)}
        item_lookup = {value: i for i, value in enumerate(self.item_ids)}
        text_rows = train["text_key"].map(text_lookup).to_numpy()
        categories = train["item_microcat_id"].map(self.category_lookup).to_numpy()
        counts = sparse.coo_matrix(
            (np.ones(len(train), dtype=np.float32), (text_rows, categories)),
            shape=(len(self.texts), len(self.category_ids)),
        ).tocsr()
        self.category_matrix = normalize_matrix(counts, norm="l1")
        same_location = train["search_location_id"].eq(train["item_location_id"])
        self.global_local_rate = float(same_location.mean())
        location_stats = (
            train.assign(same_location=same_location)
            .groupby("item_microcat_id")["same_location"]
            .agg(["sum", "count"])
        )
        local_rate = (location_stats["sum"] + 20 * same_location.mean()) / (
            location_stats["count"] + 20
        )
        self.category_local_rate = local_rate.reindex(self.category_ids).to_numpy(
            dtype=np.float32
        )

        self.word_vectorizer = TfidfVectorizer(
            tokenizer=tokenize,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=1,
            max_features=180_000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=150_000,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.word_matrix = self.word_vectorizer.fit_transform(self.texts).T.tocsr()
        self.char_matrix = self.char_vectorizer.fit_transform(self.texts).T.tocsr()

        in_corpus = train["item_id"].isin(item_lookup)
        matched = train.loc[in_corpus]
        rows = matched["text_key"].map(text_lookup).to_numpy()
        cols = matched["item_id"].map(item_lookup).to_numpy()
        self.query_items = sparse.coo_matrix(
            (np.ones(len(matched), dtype=np.float32), (rows, cols)),
            shape=(len(self.texts), len(items)),
        ).tocsr()
        self.query_items.data = np.log1p(self.query_items.data)
        self.popularity = (
            items["item_id"]
            .map(train["item_id"].value_counts())
            .fillna(0)
            .to_numpy(dtype=np.float32)
        )
        self.text_item_counts = (
            matched.groupby(["text_key", "item_id"]).size().to_dict()
        )
        self.context_item_counts = (
            matched.groupby(["context_key", "item_id"]).size().to_dict()
        )
        self.text_count = train["text_key"].value_counts().to_dict()

        filter_counts = (
            train.loc[train["search_infm_params_text"].ne("")]
            .groupby(["search_infm_params_text", "item_microcat_id"])
            .size()
        )
        filter_totals = filter_counts.groupby(level=0).sum()
        self.filter_probs = (filter_counts / filter_totals).to_dict()
        self.filter_counts = filter_totals.to_dict()
        return self

    def query_info(self, query, neighbors=40):
        text = normalize(query["search_query"])
        word = self.word_vectorizer.transform([text]) @ self.word_matrix
        char = self.char_vectorizer.transform([text]) @ self.char_matrix
        similarities = (word * 0.55 + char * 0.45).toarray().ravel()
        k = min(neighbors, len(similarities))
        indices = np.argpartition(similarities, -k)[-k:]
        indices = indices[np.lexsort((indices, -similarities[indices]))]
        weights = np.maximum(similarities[indices], 0) ** 4
        total = weights.sum()
        if total:
            weights /= total
        category_probs = np.asarray(weights @ self.category_matrix[indices]).ravel()
        memory = sparse.csr_matrix(weights.reshape(1, -1)) @ self.query_items[indices]
        return {
            "text": text,
            "category_probs": category_probs,
            "memory": memory.toarray().ravel(),
            "nearest_similarity": (
                float(similarities[indices[0]]) if len(indices) else 0.0
            ),
            "text_count": self.text_count.get(text, 0),
            "query_local_rate": float(category_probs @ self.category_local_rate),
        }

    def features(self, query, candidates, info):
        ids = self.item_ids[candidates]
        cats = self.item_categories[candidates]
        category_indices = self.item_category_indices[candidates]
        category_probs = np.zeros(len(candidates), dtype=np.float32)
        known = category_indices >= 0
        category_probs[known] = info["category_probs"][category_indices[known]]
        local_rate = np.full(len(candidates), self.global_local_rate, dtype=np.float32)
        local_rate[known] = self.category_local_rate[category_indices[known]]
        context = query_keys(pd.DataFrame([query])).iloc[0]
        text = info["text"]
        filters = query["search_infm_params_text"]
        return {
            "popularity": np.log1p(self.popularity[candidates]),
            "memory_score": info["memory"][candidates],
            "text_item_count": np.log1p(
                [self.text_item_counts.get((text, value), 0) for value in ids]
            ),
            "context_item_count": np.log1p(
                [self.context_item_counts.get((context, value), 0) for value in ids]
            ),
            "microcat_probability": category_probs,
            "category_local_rate": local_rate,
            "query_local_rate": np.full(len(candidates), info["query_local_rate"]),
            "filter_microcat_probability": np.array(
                [self.filter_probs.get((filters, cat), 0) for cat in cats]
            ),
            "filter_train_count": np.full(
                len(candidates), np.log1p(self.filter_counts.get(filters, 0))
            ),
            "query_train_count": np.full(len(candidates), np.log1p(info["text_count"])),
            "nearest_query_similarity": np.full(
                len(candidates), info["nearest_similarity"]
            ),
        }


In [ ]:
%%writefile src/avito_retrieval/geography.py
"""Мягкая география из координат объявлений и обучающих переходов поиска."""

import numpy as np
import pandas as pd


def haversine(lat_a, lon_a, lat_b, lon_b):
    """Расстояние в километрах, скаляры и массивы поддерживаются одинаково."""
    lat_a, lon_a, lat_b, lon_b = [
        np.radians(value) for value in (lat_a, lon_a, lat_b, lon_b)
    ]
    value = (
        np.sin((lat_b - lat_a) / 2) ** 2
        + np.cos(lat_a) * np.cos(lat_b) * np.sin((lon_b - lon_a) / 2) ** 2
    )
    return 12742 * np.arcsin(np.sqrt(np.clip(value, 0, 1)))


def weighted_median(values, weights):
    order = np.argsort(values, kind="stable")
    values, weights = np.asarray(values)[order], np.asarray(weights)[order]
    index = np.searchsorted(np.cumsum(weights), weights.sum() / 2, side="left")
    return float(values[index])


def coordinates(frame):
    values = frame[["item_latitude", "item_longitude"]].astype(float).to_numpy()
    valid = (
        np.isfinite(values).all(axis=1)
        & (np.abs(values[:, 0]) <= 90)
        & (np.abs(values[:, 1]) <= 180)
        & np.any(values != 0, axis=1)
    )
    values[~valid] = np.nan
    return values


class Geography:
    def fit(self, train, items, metadata_items=None):
        """В train должны быть только разрешённые обучающие пары, без eval."""
        self.item_locations = items["item_location_id"].to_numpy(copy=True)
        self.item_coordinates = coordinates(items)
        self.location_ids, self.location_indices = np.unique(
            self.item_locations, return_inverse=True
        )

        columns = ["item_id", "item_location_id", "item_latitude", "item_longitude"]
        metadata = items[columns]
        if metadata_items is not None:
            metadata = pd.concat([metadata, metadata_items[columns]], ignore_index=True)
        metadata = metadata.drop_duplicates("item_id").copy()
        metadata[["item_latitude", "item_longitude"]] = coordinates(metadata)
        centers = (
            metadata.groupby("item_location_id")[["item_latitude", "item_longitude"]]
            .median()
            .dropna()
        )
        self.centers = {
            location: point
            for location, point in zip(centers.index, centers.to_numpy())
        }

        marginal = train["item_location_id"].value_counts()
        if len(train):
            self.location_prior = (marginal / len(train)).to_dict()
        else:
            self.location_prior = (
                items["item_location_id"].value_counts(normalize=True).to_dict()
            )
        self.corpus_prior = np.array(
            [self.location_prior.get(location, 0.0) for location in self.location_ids]
        )

        # Сглаживание к общей частоте города уменьшает вес случайных единичных кликов.
        self.smoothing = 5.0
        self.transitions = {}
        pairs = train.groupby(["search_location_id", "item_location_id"]).size()
        for query_location, group in pairs.groupby(level=0, sort=False):
            self.transitions[query_location] = group.droplevel(0).to_dict()
        self.query_states = {}
        return self

    def _query_state(self, query_location):
        if query_location in self.query_states:
            return self.query_states[query_location]

        counts = self.transitions.get(query_location, {})
        train_count = sum(counts.values())
        known_locations = [location for location in counts if location in self.centers]
        point = self.centers.get(query_location)
        scope = 0.0
        if known_locations:
            points = np.array([self.centers[location] for location in known_locations])
            weights = np.array([counts[location] for location in known_locations])
            if point is None:
                # Локация поиска может обозначать область, которой нет у объявлений.
                point = np.array(
                    [
                        weighted_median(points[:, 0], weights),
                        weighted_median(points[:, 1], weights),
                    ]
                )
            distances = haversine(point[0], point[1], points[:, 0], points[:, 1])
            scope = weighted_median(distances, weights)

        observed = np.array([counts.get(location, 0) for location in self.location_ids])
        probability = (observed + self.smoothing * self.corpus_prior) / (
            train_count + self.smoothing
        )
        lift = np.divide(
            probability,
            self.corpus_prior,
            out=np.ones_like(probability),
            where=self.corpus_prior > 0,
        )
        state = {
            "point": point,
            "scope": scope,
            "train_count": train_count,
            "observed": observed,
            "probability": probability,
            "lift": lift,
        }
        self.query_states[query_location] = state
        return state

    def _distances(self, state, candidate_indices):
        points = self.item_coordinates[candidate_indices]
        has_center = np.isfinite(points).all(axis=1)
        if state["point"] is None:
            return np.zeros(len(points)), np.zeros(len(points), dtype=bool)
        point = state["point"]
        distances = haversine(point[0], point[1], points[:, 0], points[:, 1])
        return np.where(has_center, distances, 0.0), has_center

    def features(self, search_location_id, candidate_indices):
        candidate_indices = np.asarray(candidate_indices, dtype=np.int64)
        state = self._query_state(search_location_id)
        distances, has_center = self._distances(state, candidate_indices)
        locations = self.location_indices[candidate_indices]
        size = len(candidate_indices)
        result = {
            "same_location": self.item_locations[candidate_indices]
            == search_location_id,
            "log_distance": np.log1p(distances),
            "has_center": has_center,
            "transition_probability": state["probability"][locations],
            "transition_lift": state["lift"][locations],
            "scope_km": np.full(size, state["scope"]),
            "train_count": np.full(size, state["train_count"]),
        }
        return {name: values.astype(np.float32) for name, values in result.items()}

    def affinity(self, search_location_id):
        """Множитель текстовой близости в [0, 1], без жёсткого отсечения городов."""
        state = self._query_state(search_location_id)
        if state["point"] is None:
            return np.ones(len(self.item_locations), dtype=np.float32)

        indices = np.arange(len(self.item_locations))
        distances, has_center = self._distances(state, indices)
        scale = 25.0 + min(state["scope"], 150.0)
        nearby = 1.0 / (1.0 + distances / scale)
        nearby[~has_center] = 0.5

        probability = state["probability"]
        relative = probability / max(float(probability.max(initial=0)), 1e-12)
        lift = state["lift"]
        support = state["observed"] / (state["observed"] + self.smoothing)
        transition = support * (0.7 * np.sqrt(relative) + 0.3 * lift / (1 + lift))
        affinity = np.maximum(nearby, 0.85 * transition[self.location_indices])

        # Общероссийский поиск не должен становиться московским из-за медианы кликов.
        floor = 0.9 if state["scope"] > 300 else 0.02
        affinity = floor + (1 - floor) * affinity
        affinity[self.item_locations == search_location_id] = 1.0
        return np.clip(affinity, 0, 1).astype(np.float32)


In [ ]:
%%writefile src/avito_retrieval/candidates.py
"""Собираем кандидатов из независимых источников и признаки для отбора 50."""

import time
import os
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse

from avito_retrieval.lexical import _topk
from avito_retrieval.text import normalize, tokenize


def top_indices(scores, count, positive_only=True):
    keep = np.isfinite(scores)
    if positive_only:
        keep &= scores > 0
    indices = np.flatnonzero(keep)
    return _topk(indices, scores[indices], count)[0].astype(np.int32)


class CandidateBuilder:
    def __init__(self, items, lexical, behavior, geography, item_embeddings=None):
        if list(items["item_id"]) != list(lexical.item_ids_):
            raise ValueError("Lexical index and corpus have different item order")
        if list(items["item_id"]) != list(behavior.item_ids):
            raise ValueError("Behavior index and corpus have different item order")
        self.items = items
        self.lexical = lexical
        bm25 = lexical.matrices_["bm25"]
        self.term_presence = sparse.csr_matrix(
            (np.ones_like(bm25.data), bm25.indices, bm25.indptr), shape=bm25.shape
        )
        self.behavior = behavior
        self.geography = geography
        self.item_embeddings = item_embeddings
        self.titles = items["item_title_raw"].fillna("").map(normalize).to_numpy()
        self.title_tokens = [set(tokenize(value)) for value in self.titles]
        self.params = (
            items["item_infm_params_text"]
            .fillna("")
            .map(lambda value: " " + " ".join(tokenize(value)) + " ")
            .to_numpy()
        )
        self.numeric = {}
        for name in ["item_price", "item_rating_reviews_count", "item_rating"]:
            values = (
                pd.to_numeric(items[name], errors="coerce")
                .fillna(0)
                .to_numpy(dtype=np.float32)
            )
            if name != "item_rating":
                values = np.log1p(np.maximum(values, 0))
            self.numeric[name] = values
        for name in ["item_is_phone_hidden", "item_is_message_forbidden"]:
            self.numeric[name] = items[name].fillna(False).to_numpy(dtype=np.float32)
        self.numeric["title_length"] = np.array(
            [len(value) for value in self.titles], dtype=np.float32
        )
        self.numeric["description_length"] = (
            items["item_description_raw"]
            .fillna("")
            .str.len()
            .to_numpy(dtype=np.float32)
        )
        self.affinity = lru_cache(maxsize=64)(geography.affinity)

    def _lexical_scores(self, queries):
        scores = {}
        texts = queries["search_query"].fillna("").tolist()
        for name, vectorizer in self.lexical.vectorizers_.items():
            vectors = vectorizer.transform(texts)
            if name == "bm25":
                vectors.data.fill(1)
                coverage = (vectors @ self.term_presence).toarray()
                lengths = np.array([max(len(set(tokenize(text))), 1) for text in texts])
                scores["full_coverage"] = coverage / lengths[:, None]
            scores[name] = (vectors @ self.lexical.matrices_[name]).toarray()
        context = (
            queries["search_query"].fillna("")
            + " "
            + queries["search_infm_params_text"].fillna("")
        ).tolist()
        vectors = self.lexical.vectorizers_["word"].transform(context)
        scores["context_word"] = (vectors @ self.lexical.matrices_["word"]).toarray()
        return scores

    def _one_query(self, query, raw_scores, dense_scores, allowed_n_items):
        n_items = allowed_n_items or len(self.items)
        geo = self.affinity(query["search_location_id"])[:n_items]
        raw = {name: values[:n_items] for name, values in raw_scores.items()}
        info = self.behavior.query_info(query)
        memory = info["memory"][:n_items]
        channels = {}
        geo_limits = {"word": 220, "bm25": 500, "char": 350}
        for name in ["word", "bm25", "char"]:
            channels[name] = (raw[name], 100)
            channels[f"geo_{name}"] = (
                raw[name] * (0.015 + 0.985 * geo) ** 0.7,
                geo_limits[name],
            )
        channels["geo_context"] = (raw["context_word"] * geo**0.7, 220)
        channels["memory"] = (memory, 60)
        channels["geo_memory"] = (memory * geo, 120)
        if dense_scores is not None:
            for name, values in dense_scores.items():
                raw[name] = values[:n_items]
                channels[name] = (raw[name], 120)
                # У E5 близкие косинусы, поправка сохраняет этот масштаб.
                channels[f"geo_{name}"] = (
                    raw[name] + 0.055 * np.log(0.005 + 0.995 * geo),
                    300,
                )

        sources = {
            name: top_indices(values, count, positive_only="dense" not in name)
            for name, (values, count) in channels.items()
        }
        candidates = np.unique(np.concatenate(list(sources.values())))
        if len(candidates) < 50:
            fallback = top_indices(
                (1 + self.numeric["item_rating_reviews_count"][:n_items]) * geo, 50
            )
            candidates = np.union1d(candidates, fallback)
        result = {"item_idx": candidates.astype(np.int32)}
        for name, values in raw.items():
            result[name] = values[candidates]
            result[f"relative_{name}"] = values[candidates] / max(
                float(values.max()), 1e-6
            )
        for name, indices in sources.items():
            ranks = np.zeros(len(candidates), dtype=np.float32)
            ranks[np.searchsorted(candidates, indices)] = 1 / np.sqrt(
                1 + np.arange(len(indices))
            )
            result[f"rank_{name}"] = ranks
        result["geo_affinity"] = geo[candidates]
        result.update(self.geography.features(query["search_location_id"], candidates))
        result.update(self.behavior.features(query, candidates, info))
        result.update(
            {name: values[candidates] for name, values in self.numeric.items()}
        )
        query_tokens = set(tokenize(query["search_query"]))
        query_text = normalize(query["search_query"])
        filter_tokens = set(tokenize(query["search_infm_params_text"]))
        size = len(candidates)
        result["title_coverage"] = np.array(
            [
                len(query_tokens & self.title_tokens[i]) / max(len(query_tokens), 1)
                for i in candidates
            ]
        )
        result["title_precision"] = np.array(
            [
                len(query_tokens & self.title_tokens[i])
                / max(len(self.title_tokens[i]), 1)
                for i in candidates
            ]
        )
        result["title_exact_phrase"] = np.array(
            [bool(query_text) and query_text in self.titles[i] for i in candidates]
        )
        result["filter_coverage"] = np.array(
            [
                sum(f" {token} " in self.params[i] for token in filter_tokens)
                / max(len(filter_tokens), 1)
                for i in candidates
            ]
        )
        result["query_words"] = np.full(size, len(query_tokens))
        result["query_length"] = np.full(size, len(query_text))
        result["has_filters"] = np.full(size, bool(filter_tokens))
        result["delivery_search"] = np.full(size, query["search_is_delivery_search"])
        result["category_match"] = (
            self.items["item_category_id"].to_numpy()[candidates]
            == query["search_category"]
        )
        result["lexical_heuristic"] = (
            result["relative_word"]
            + 0.7 * result["relative_bm25"]
            + 0.5 * result["relative_char"]
        ) * result["geo_affinity"] ** 0.7
        if dense_scores is not None:
            result["hybrid_heuristic"] = (
                result["dense"]
                + 0.055 * np.log(0.005 + 0.995 * result["geo_affinity"])
                + 0.035 * result["lexical_heuristic"]
            )
        return {
            name: values.astype(np.float32) if name != "item_idx" else values
            for name, values in result.items()
        }

    def write_features(
        self,
        queries,
        output_path,
        truth=None,
        query_embeddings=None,
        allowed_n_items=None,
        batch_size=16,
    ):
        ids = self.items["item_id"].to_numpy()
        targets = (
            {}
            if truth is None
            else truth.groupby("query_id")["item_id"].agg(set).to_dict()
        )
        writer = None
        output_path = Path(output_path)
        temporary_path = output_path.with_suffix(output_path.suffix + ".part")
        start_time = time.monotonic()
        total_rows = 0
        try:
            for start in range(0, len(queries), batch_size):
                batch = queries.iloc[start : start + batch_size]
                lexical_scores = self._lexical_scores(batch)
                dense_scores = None
                if query_embeddings is not None:
                    dense_scores = {
                        name: values[start : start + len(batch)]
                        @ self.item_embeddings.T
                        for name, values in query_embeddings.items()
                    }
                frames = []
                for offset, (_, query) in enumerate(batch.iterrows()):
                    one_dense = (
                        None
                        if dense_scores is None
                        else {
                            name: values[offset]
                            for name, values in dense_scores.items()
                        }
                    )
                    result = self._one_query(
                        query,
                        {key: value[offset] for key, value in lexical_scores.items()},
                        one_dense,
                        allowed_n_items,
                    )
                    frame = pd.DataFrame(result)
                    frame.insert(0, "qid", np.int32(start + offset))
                    correct = targets.get(query["query_id"], set())
                    frame["label"] = np.array(
                        [
                            value in correct
                            for value in ids[frame["item_idx"].to_numpy()]
                        ],
                        dtype=np.int8,
                    )
                    frames.append(frame)
                table = pa.Table.from_pandas(
                    pd.concat(frames, ignore_index=True), preserve_index=False
                )
                if writer is None:
                    writer = pq.ParquetWriter(
                        temporary_path, table.schema, compression="zstd"
                    )
                writer.write_table(table)
                total_rows += table.num_rows
                if start == 0 or (start // batch_size + 1) % 10 == 0:
                    elapsed = time.monotonic() - start_time
                    done = start + len(batch)
                    print(
                        f"{done}/{len(queries)} запросов, {total_rows:,} кандидатов, {elapsed:.1f}с",
                        flush=True,
                    )
        finally:
            if writer is not None:
                writer.close()
        os.replace(temporary_path, output_path)


In [ ]:
%%writefile scripts/build_features.py
"""Готовим признаки кандидатов, обучение и выбор конфигурации идут отдельно."""

import argparse
import hashlib
import json
from pathlib import Path

import joblib
import pandas as pd
from threadpoolctl import threadpool_limits

from avito_retrieval.behavior import BehaviorIndex
from avito_retrieval.candidates import CandidateBuilder
from avito_retrieval.geography import Geography


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--mode", choices=["validation", "benchmark"], default="validation"
    )
    parser.add_argument("--dense", action="store_true")
    parser.add_argument(
        "--plain-query",
        action="store_true",
        help="Добавить E5-представление запроса без фильтров",
    )
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--limit", type=int)
    args = parser.parse_args()
    if args.plain_query and not args.dense:
        parser.error("--plain-query requires --dense")
    threadpool_limits(limits=3)
    items = pd.read_parquet("data/processed/corpus.parquet")
    if args.mode == "validation":
        train_path = "data/processed/retrieval_train.parquet"
        train = pd.read_parquet(train_path)
        queries = pd.read_parquet("data/processed/eval_queries.parquet")
        truth = pd.read_parquet("data/processed/eval_truth.parquet")
        prefix = None
    else:
        train_path = "data/raw/train.parquet"
        columns = [
            "search_query",
            "search_location_id",
            "search_is_delivery_search",
            "search_infm_params_text",
            "search_category",
            "item_id",
            "item_microcat_id",
            "item_location_id",
        ]
        train = pd.read_parquet("data/raw/train.parquet", columns=columns)
        queries = pd.read_parquet("data/raw/benchmark_queries.parquet")
        truth = None
        prefix = len(
            pd.read_parquet("data/raw/benchmark_items.parquet", columns=["item_id"])
        )
    queries = queries.iloc[: args.limit] if args.limit else queries
    knowledge_path = Path(f"artifacts/knowledge_{args.mode}.joblib")
    metadata_path = knowledge_path.with_suffix(".json")
    fingerprint = {
        "train_sha256": file_hash(train_path),
        "corpus_sha256": file_hash("data/processed/corpus.parquet"),
        "behavior_code_sha256": file_hash("src/avito_retrieval/behavior.py"),
        "geography_code_sha256": file_hash("src/avito_retrieval/geography.py"),
    }
    if (
        knowledge_path.exists()
        and metadata_path.exists()
        and json.loads(metadata_path.read_text()) == fingerprint
    ):
        behavior, geography = joblib.load(knowledge_path)
    else:
        print("Обучаем статистики запросов и географии", flush=True)
        behavior = BehaviorIndex().fit(train, items)
        geography = Geography().fit(train, items)
        joblib.dump((behavior, geography), knowledge_path)
        metadata_path.write_text(json.dumps(fingerprint, indent=2) + "\n")
    print("Загружаем текстовый индекс", flush=True)
    lexical = joblib.load("artifacts/lexical.joblib", mmap_mode="r")
    item_embeddings = query_embeddings = None
    if args.dense:
        from avito_retrieval.dense import load_embeddings

        item_embeddings, item_ids = load_embeddings(Path("artifacts/dense"), "items")
        query_name = (
            "eval_queries" if args.mode == "validation" else "benchmark_queries"
        )
        query_embeddings, query_ids = load_embeddings(
            Path("artifacts/dense"), query_name
        )
        if (
            list(item_ids) != items["item_id"].tolist()
            or list(query_ids[: len(queries)]) != queries["query_id"].tolist()
        ):
            raise ValueError("Embedding IDs differ from source data")
        query_embeddings = {"dense": query_embeddings[: len(queries)]}
        if args.plain_query:
            plain, plain_ids = load_embeddings(
                Path("artifacts/dense"), query_name + "_plain"
            )
            if list(plain_ids[: len(queries)]) != queries["query_id"].tolist():
                raise ValueError("Plain query embedding IDs differ from source data")
            query_embeddings["dense_plain"] = plain[: len(queries)]
    builder = CandidateBuilder(items, lexical, behavior, geography, item_embeddings)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    print("Собираем кандидатов и признаки", flush=True)
    builder.write_features(queries, args.output, truth, query_embeddings, prefix)
    query_path = (
        "data/processed/eval_queries.parquet"
        if args.mode == "validation"
        else "data/raw/benchmark_queries.parquet"
    )
    metadata = {
        "mode": args.mode,
        "dense": args.dense,
        "plain_query": args.plain_query,
        "query_count": len(queries),
        "allowed_n_items": prefix or len(items),
        "queries_sha256": file_hash(query_path),
        "knowledge": fingerprint,
        "candidate_code_sha256": file_hash("src/avito_retrieval/candidates.py"),
        "features_sha256": file_hash(args.output),
    }
    args.output.with_suffix(".metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )


if __name__ == "__main__":
    main()


In [ ]:
run_stage("train_features")


## 9. Обучение моделей

Ниже находятся реализация метрики и обучение CatBoost. Первая модель использует прямые признаки истории, вторая исключает шесть таких признаков. Обе обучаются заново на fit и dev. Идентификаторы в признаки не включаются.

In [ ]:
%%writefile src/avito_retrieval/evaluation.py
"""Macro Recall с полной разметкой, включая не найденные генератором объявления."""

from pathlib import Path

import numpy as np
import pandas as pd


RESERVED_COLUMNS = {
    "qid",
    "item_idx",
    "label",
    "query_id",
    "item_id",
    "split",
    "target",
}


def load_split_queries(path: str | Path, splits: list[str]) -> pd.DataFrame:
    # Позиция qid привязана к исходному файлу и не меняется после фильтрации.
    identities = pd.read_parquet(path, columns=["query_id", "split"])
    identities["qid"] = np.arange(len(identities), dtype=np.int32)
    selected = pd.read_parquet(path, filters=[("split", "in", splits)])
    return selected.merge(
        identities[["query_id", "qid"]], on="query_id", validate="one_to_one"
    )


def feature_columns(frame: pd.DataFrame) -> list[str]:
    names = [name for name in frame if name not in RESERVED_COLUMNS]
    if not names:
        raise ValueError("no model features found")
    invalid = [name for name in names if not pd.api.types.is_numeric_dtype(frame[name])]
    if invalid:
        raise ValueError(f"model features must be numeric: {invalid}")
    return names


def reference_empty_fraction(path: str | Path) -> float:
    """Вычислить долю запросов без фильтров по признакам, без тестовой разметки."""
    frame = pd.read_parquet(path, columns=["search_infm_params_text"])
    if frame.empty:
        raise ValueError("reference queries must not be empty")
    return float(
        frame["search_infm_params_text"]
        .map(lambda value: not isinstance(value, str) or not value.strip())
        .mean()
    )


def evaluate_candidates(
    candidates: pd.DataFrame,
    queries: pd.DataFrame,
    truth: pd.DataFrame,
    corpus: pd.DataFrame,
    score_column="score",
    ks=(10, 50, 100),
    reference_empty_fraction=None,
) -> tuple[dict, pd.DataFrame]:
    """Вернуть сводку и строки запросов, порядок ничьих фиксирован по item_idx.

    queries содержит qid исходного eval_queries и query_id. Пустые пулы остаются
    в среднем с нулевым Recall. label из candidates в расчёте не используется.
    """
    ks = tuple(sorted(set(ks)))
    if not ks or any(not isinstance(k, (int, np.integer)) or k < 1 for k in ks):
        raise ValueError("ks must contain positive integer cutoffs")
    if queries["qid"].duplicated().any() or queries["query_id"].duplicated().any():
        raise ValueError("queries must have unique qid and query_id")
    if candidates[["qid", "item_idx"]].duplicated().any():
        raise ValueError("duplicate query-item candidate")
    if not set(candidates["qid"]).issubset(set(queries["qid"])):
        raise ValueError("candidate qid outside selected queries")
    if not pd.api.types.is_integer_dtype(candidates["item_idx"]):
        raise ValueError("item_idx must be integer")
    indices = candidates["item_idx"].to_numpy()
    if np.any(indices < 0) or np.any(indices >= len(corpus)):
        raise ValueError("candidate item_idx outside corpus")
    values = candidates[score_column].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError("candidate scores must be finite")
    if corpus["item_id"].duplicated().any():
        raise ValueError("corpus item_id must be unique")

    truth_by_query = truth.groupby("query_id", sort=False)["item_id"].agg(set).to_dict()
    seen_by_query = None
    if "seen_item" in truth:
        seen_by_query = (
            truth.groupby("query_id", sort=False)["seen_item"]
            .agg(
                lambda values: (
                    "all_seen"
                    if values.all()
                    else "mixed" if values.any() else "all_unseen"
                )
            )
            .to_dict()
        )
    order = np.lexsort((indices, -values, candidates["qid"].to_numpy()))
    ranked = candidates.iloc[order]
    ranked_by_query = {
        int(qid): group["item_idx"].to_numpy()
        for qid, group in ranked.groupby("qid", sort=False)
    }
    item_ids = corpus["item_id"].to_numpy()
    rows = []
    for query in queries.to_dict("records"):
        qid, query_id = query["qid"], query["query_id"]
        relevant = truth_by_query.get(query_id, set())
        if not relevant:
            raise ValueError(f"query {query_id!r} has no ground-truth relevant items")
        found = item_ids[ranked_by_query.get(qid, np.empty(0, dtype=np.int32))]
        hits = np.fromiter((item_id in relevant for item_id in found), dtype=np.int32)
        row = {
            "qid": qid,
            "query_id": query_id,
            "relevant_count": len(relevant),
            "candidate_count": len(found),
            "recall@union": float(hits.sum() / len(relevant)),
        }
        row.update({f"recall@{k}": float(hits[:k].sum() / len(relevant)) for k in ks})
        for name in ("regime", "seen_text", "seen_context"):
            if name in query:
                row[name] = query[name]
        filters = query.get("search_infm_params_text", "")
        row["filter_empty"] = not isinstance(filters, str) or not filters.strip()
        if seen_by_query is not None:
            row["seen_item_group"] = seen_by_query[query_id]
        rows.append(row)
    per_query = pd.DataFrame(rows)
    metric_names = [f"recall@{k}" for k in ks] + ["recall@union"]
    summary = {"queries": len(queries)}
    summary.update(
        {
            name: float(per_query[name].mean()) if len(per_query) else 0.0
            for name in metric_names
        }
    )
    summary["candidate_count_mean"] = (
        float(per_query["candidate_count"].mean()) if len(per_query) else 0.0
    )
    summary["groups"] = {}
    for column in (
        "regime",
        "seen_text",
        "seen_context",
        "filter_empty",
        "seen_item_group",
    ):
        if column in per_query:
            summary["groups"][column] = {
                str(value): {
                    "queries": len(group),
                    **{name: float(group[name].mean()) for name in metric_names},
                }
                for value, group in per_query.groupby(column, dropna=False)
            }
    if reference_empty_fraction is not None:
        fraction = float(reference_empty_fraction)
        if not 0 <= fraction <= 1:
            raise ValueError("reference_empty_fraction must be between 0 and 1")
        summary["reference_empty_fraction"] = fraction
        groups = summary["groups"].get("filter_empty", {})
        for name in metric_names:
            weighted = 0.0
            for empty, weight in (("True", fraction), ("False", 1 - fraction)):
                if not weight:
                    continue
                if empty not in groups:
                    weighted = None
                    break
                weighted += weight * groups[empty][name]
            # Отсутствующую в dev группу нельзя заменить нулём или выдуманной оценкой.
            summary[f"poststratified_{name}"] = weighted
    return summary, per_query


In [ ]:
%%writefile scripts/train_ranker.py
"""Обучить ранжирование на fit и выбрать число деревьев по Recall@50 на dev."""

import argparse
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRanker, Pool

from avito_retrieval.evaluation import (
    evaluate_candidates,
    feature_columns,
    load_split_queries,
    reference_empty_fraction,
)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--features", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument(
        "--queries", type=Path, default=Path("data/processed/eval_queries.parquet")
    )
    parser.add_argument(
        "--truth", type=Path, default=Path("data/processed/eval_truth.parquet")
    )
    parser.add_argument(
        "--corpus", type=Path, default=Path("data/processed/corpus.parquet")
    )
    parser.add_argument(
        "--loss", choices=["QuerySoftMax", "Logloss"], default="QuerySoftMax"
    )
    parser.add_argument("--iterations", type=int, default=600)
    parser.add_argument("--depth", type=int, default=6)
    parser.add_argument("--learning-rate", type=float, default=0.07)
    parser.add_argument("--checkpoint-period", type=int, default=50)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--drop-features", default="", help="Comma-separated feature names"
    )
    parser.add_argument("--fit-splits", default="fit", help="fit or fit,dev")
    parser.add_argument(
        "--fixed-iterations",
        action="store_true",
        help="Use the requested tree count without dev selection",
    )
    parser.add_argument(
        "--reference-queries",
        type=Path,
        default=Path("data/raw/benchmark_queries.parquet"),
    )
    args = parser.parse_args()
    if args.iterations < 1 or args.checkpoint_period < 1:
        parser.error("iterations and checkpoint-period must be positive")
    fit_splits = [value.strip() for value in args.fit_splits.split(",")]
    if set(fit_splits) not in ({"fit"}, {"fit", "dev"}) or len(fit_splits) != len(
        set(fit_splits)
    ):
        parser.error(
            "fit-splits must be fit or fit,dev, audit is never a training split"
        )
    if "dev" in fit_splits and not args.fixed_iterations:
        parser.error("including dev in training requires --fixed-iterations")
    start = time.monotonic()
    queries = load_split_queries(
        args.queries, fit_splits if args.fixed_iterations else ["fit", "dev"]
    )
    dev_queries = queries.loc[queries["split"].eq("dev")].copy()
    fit_qids = set(queries.loc[queries["split"].isin(fit_splits), "qid"])
    dev_qids = set() if args.fixed_iterations else set(dev_queries["qid"])
    if not fit_qids or (not args.fixed_iterations and not dev_qids):
        raise ValueError("both fit and dev queries are required")
    # Audit-строки не поступают ни в обучение, ни в подбор числа деревьев.
    features = pd.read_parquet(
        args.features, filters=[("qid", "in", sorted(fit_qids | dev_qids))]
    )
    expected_qids = fit_qids | dev_qids
    if set(features["qid"]) != expected_qids:
        missing = expected_qids - set(features["qid"])
        raise ValueError(
            f"candidate qid coverage mismatch: {len(missing)} missing, examples={sorted(missing)[:5]}"
        )
    for name in ("qid", "item_idx"):
        if not pd.api.types.is_integer_dtype(features[name]):
            raise ValueError(f"{name} must be integer")
    if features[["qid", "item_idx"]].duplicated().any():
        raise ValueError("duplicate query-item candidate")
    names = feature_columns(features)
    dropped = {name.strip() for name in args.drop_features.split(",") if name.strip()}
    unknown = dropped - set(names)
    if unknown:
        raise ValueError(f"unknown drop-features: {sorted(unknown)}")
    names = [name for name in names if name not in dropped]
    if not names:
        raise ValueError("drop-features removed every model feature")
    invalid = [
        name for name in names if not np.isfinite(features[name].to_numpy()).all()
    ]
    if invalid:
        raise ValueError(f"non-finite model features: {invalid}")
    if not features["label"].isin([0, 1]).all():
        raise ValueError("label must be binary")
    fit = features.loc[features["qid"].isin(fit_qids)].sort_values(["qid", "item_idx"])
    dev = features.loc[features["qid"].isin(dev_qids)].sort_values(["qid", "item_idx"])
    del features
    corpus = pd.read_parquet(args.corpus, columns=["item_id"])
    for frame in (fit, dev):
        if not frame["item_idx"].between(0, len(corpus) - 1).all():
            raise ValueError("candidate item_idx outside corpus")
    parameters = dict(
        loss_function=args.loss,
        iterations=args.iterations,
        depth=args.depth,
        learning_rate=args.learning_rate,
        random_seed=args.seed,
        thread_count=args.threads,
        allow_writing_files=False,
    )
    if args.loss == "QuerySoftMax":
        model = CatBoostRanker(**parameters)
        fit_pool = Pool(fit[names], label=fit["label"], group_id=fit["qid"])
    else:
        model = CatBoostClassifier(**parameters)
        fit_pool = Pool(fit[names], label=fit["label"])
    dev_pool = Pool(dev[names]) if not args.fixed_iterations else None
    print(
        json.dumps(
            {"fit_rows": len(fit), "dev_rows": len(dev), "features": len(names)}
        ),
        flush=True,
    )
    model.fit(fit_pool, verbose=100)
    del fit, fit_pool
    config = {
        key: str(value) if isinstance(value, Path) else value
        for key, value in vars(args).items()
    }
    config.update(
        {
            "fit_query_count": len(fit_qids),
            "dev_query_count": len(dev_qids),
            "dropped_features": sorted(dropped),
        }
    )
    if args.fixed_iterations:
        # Финальный retrain использует заранее выбранное число деревьев, без повторного подбора.
        config.update(
            {
                "selected_iteration": model.tree_count_,
                "selected_dev_recall@50": None,
                "seconds": round(time.monotonic() - start, 3),
            }
        )
        save_model(model, names, config, [], args.output)
        print(json.dumps(config, indent=2), flush=True)
        return
    truth = pd.read_parquet(args.truth, filters=[("split", "=", "dev")])
    fraction = reference_empty_fraction(args.reference_queries)
    predictions = model.staged_predict(
        dev_pool,
        eval_period=args.checkpoint_period,
        thread_count=args.threads,
        **({"prediction_type": "RawFormulaVal"} if args.loss == "Logloss" else {}),
    )
    candidates = dev[["qid", "item_idx"]].copy()
    checkpoints = []
    best_recall, best_iteration = -1.0, 0
    for number, scores in enumerate(predictions, start=1):
        iteration = min(number * args.checkpoint_period, model.tree_count_)
        candidates["score"] = np.asarray(scores).reshape(-1)
        summary, _ = evaluate_candidates(
            candidates, dev_queries, truth, corpus, reference_empty_fraction=fraction
        )
        record = {"iteration": iteration, **summary}
        checkpoints.append(record)
        print(
            json.dumps({"iteration": iteration, "dev_recall@50": summary["recall@50"]}),
            flush=True,
        )
        # Равный Recall оставляет более простую модель с меньшим числом деревьев.
        if summary["recall@50"] > best_recall:
            best_recall, best_iteration = summary["recall@50"], iteration
    if best_iteration == 0:
        raise RuntimeError("CatBoost produced no evaluation checkpoints")
    model.shrink(ntree_end=best_iteration)
    config.update(
        {
            "selected_iteration": best_iteration,
            "selected_dev_recall@50": best_recall,
            "reference_empty_fraction": fraction,
            "seconds": round(time.monotonic() - start, 3),
        }
    )
    scores = model.predict(
        dev_pool,
        thread_count=args.threads,
        **({"prediction_type": "RawFormulaVal"} if args.loss == "Logloss" else {}),
    )
    candidates["score"] = np.asarray(scores).reshape(-1)
    selected_summary, per_query = evaluate_candidates(
        candidates, dev_queries, truth, corpus, reference_empty_fraction=fraction
    )
    if not np.isclose(selected_summary["recall@50"], best_recall, rtol=0, atol=1e-12):
        raise RuntimeError("shrunk model does not reproduce selected Recall@50")
    save_model(model, names, config, checkpoints, args.output)
    per_query.to_parquet(args.output.with_suffix(".dev_queries.parquet"), index=False)
    print(json.dumps(config, indent=2), flush=True)


def save_model(model, names, config, checkpoints, output):
    output.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(output)
    output.with_suffix(".feature_names.json").write_text(
        json.dumps(names, indent=2) + "\n"
    )
    output.with_suffix(".train_config.json").write_text(
        json.dumps(config, indent=2) + "\n"
    )
    output.with_suffix(".evaluation.json").write_text(
        json.dumps(
            {
                "selected_iteration": config["selected_iteration"],
                "checkpoints": checkpoints,
            },
            indent=2,
        )
        + "\n"
    )


if __name__ == "__main__":
    main()


In [ ]:
config = json.loads((WORK / "configs/final.json").read_text())
print(json.dumps(config["models"], ensure_ascii=False, indent=2))
run_stage("train")


## 10. Проверить качество на audit

Для каждого запроса берётся доля найденных положительных объявлений, затем среднее по запросам:

$$
\operatorname{Recall@50} = \frac{1}{|Q|}\sum_{q\in Q}\frac{|T_{50}(q)\cap R(q)|}{|R(q)|}.
$$

Положительные объявления, не попавшие в кандидаты, остаются в знаменателе.
Проверяется заранее выбранная смесь с весами 0,5 и 0,5. По этому audit веса не меняются.

In [ ]:
%%writefile src/avito_retrieval/submission.py
"""Проверка и сохранение ответа без потери регистра и ведущих нулей."""

from __future__ import annotations

import argparse
import json
import os
import re
import tempfile
from collections.abc import Iterable, Sequence
from numbers import Integral
from pathlib import Path

import pandas as pd


ITEM_ID_PATTERN = re.compile(r"[0-9a-f]{16}")


def _query_ids(frame: pd.DataFrame, source: str) -> list[str]:
    if list(frame.columns).count("query_id") != 1:
        raise ValueError(f"{source}: expected one query_id column")
    values = frame["query_id"].tolist()
    for row, value in enumerate(values):
        if not isinstance(value, str):
            raise ValueError(f"{source}, row {row}: query_id must be a string")
        if len(value) != 16 or any(char.isspace() for char in value):
            raise ValueError(
                f"{source}, row {row}: query_id must contain exactly 16 non-space characters"
            )
    if len(values) != len(set(values)):
        raise ValueError(f"{source}: duplicate query_id")
    return values


def _item_ids(items: pd.DataFrame) -> set[str]:
    if list(items.columns).count("item_id") != 1:
        raise ValueError("items: expected one item_id column")
    values = items["item_id"].tolist()
    for row, value in enumerate(values):
        if not isinstance(value, str) or ITEM_ID_PATTERN.fullmatch(value) is None:
            raise ValueError(
                f"items, row {row}: item_id must be a string of 16 lowercase hex characters"
            )
    if len(values) != len(set(values)):
        raise ValueError("items: duplicate item_id")
    return set(values)


def validate_submission(
    answer: pd.DataFrame,
    queries: pd.DataFrame,
    items: pd.DataFrame,
    max_candidates: int = 50,
) -> dict:
    """Проверить формат и покрытие запросов, вернуть статистику кандидатов.

    Пустая строка answer разрешена. Пропуски, приведение чисел к строкам и
    исправление регистра намеренно не поддерживаются: потерянный ID не восстановить.
    Все ошибки входных данных приводят к ValueError.
    """
    if isinstance(max_candidates, bool) or not isinstance(max_candidates, Integral):
        raise ValueError("max_candidates must be a non-negative integer")
    if max_candidates < 0:
        raise ValueError("max_candidates must be a non-negative integer")
    if len(answer.columns) != 2 or set(answer.columns) != {"query_id", "answer"}:
        raise ValueError("answer: expected exactly the columns query_id and answer")

    expected = set(_query_ids(queries, "queries"))
    actual = set(_query_ids(answer, "answer"))
    if expected != actual:
        missing, extra = expected - actual, actual - expected
        raise ValueError(
            f"query coverage mismatch: {len(missing)} missing, {len(extra)} extra, "
            f"missing examples={sorted(missing)[:3]}, extra examples={sorted(extra)[:3]}"
        )
    corpus = _item_ids(items)
    counts = []
    for row, value in enumerate(answer["answer"].tolist()):
        if not isinstance(value, str):
            raise ValueError(
                f"answer, row {row}: answer must be a string, not a missing value"
            )

        # split(" ") сохраняет лишние пробелы как пустые токены, чтобы их заметить.
        candidates = value.split(" ") if value else []
        if len(candidates) > max_candidates:
            raise ValueError(
                f"answer, row {row}: more than {max_candidates} candidates"
            )
        for item_id in candidates:
            if ITEM_ID_PATTERN.fullmatch(item_id) is None:
                raise ValueError(
                    f"answer, row {row}: invalid item_id {item_id!r}, "
                    "use lowercase 16-character hex IDs separated by one space"
                )
        if len(candidates) != len(set(candidates)):
            raise ValueError(f"answer, row {row}: duplicate item_id")
        unknown = set(candidates) - corpus
        if unknown:
            raise ValueError(
                f"answer, row {row}: unknown item_id {sorted(unknown)[:3]}"
            )
        counts.append(len(candidates))

    return {
        "rows": len(counts),
        "empty_answers": counts.count(0),
        "candidates_total": sum(counts),
        "candidates_min": min(counts, default=0),
        "candidates_max": max(counts, default=0),
        "candidates_mean": sum(counts) / len(counts) if counts else 0.0,
        "max_candidates": int(max_candidates),
    }


def write_submission(
    query_ids: Iterable[str],
    predictions: Iterable[Sequence[str]],
    output_path: str | Path,
    items: pd.DataFrame,
) -> dict:
    """Сохранить проверенный UTF-8 CSV, порядок кандидатов оставить прежним.

    Полное покрытие здесь проверяется относительно переданного query_ids.
    Для независимой проверки относительно бенчмарка используйте CLI.
    """
    ids = list(query_ids)
    prediction_rows = list(predictions)
    if len(ids) != len(prediction_rows):
        raise ValueError("query_ids and predictions must have the same number of rows")
    joined = []
    for row, candidates in enumerate(prediction_rows):
        if isinstance(candidates, (str, bytes)) or not isinstance(candidates, Iterable):
            raise ValueError(
                f"predictions, row {row}: expected a sequence of item_id strings"
            )
        candidate_ids = list(candidates)
        if any(not isinstance(item_id, str) for item_id in candidate_ids):
            raise ValueError(f"predictions, row {row}: item_id must be a string")
        if any(ITEM_ID_PATTERN.fullmatch(item_id) is None for item_id in candidate_ids):
            raise ValueError(f"predictions, row {row}: invalid item_id")
        joined.append(" ".join(candidate_ids))

    answer = pd.DataFrame({"query_id": ids, "answer": joined})
    queries = pd.DataFrame({"query_id": ids})
    report = validate_submission(answer, queries, items)
    destination = Path(output_path)
    destination.parent.mkdir(parents=True, exist_ok=True)

    # При сбое записи уже готовый ответ останется целым.
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="",
            dir=destination.parent,
            prefix=f".{destination.name}.",
            suffix=".tmp",
            delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            answer.to_csv(handle, index=False)
        os.replace(temporary_path, destination)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
    return {**report, "output_path": str(destination)}


def main() -> None:
    parser = argparse.ArgumentParser(description="Validate an Avito contest submission")
    parser.add_argument("--answer", type=Path, required=True)
    parser.add_argument("--queries", type=Path, required=True)
    parser.add_argument("--items", type=Path, required=True)
    args = parser.parse_args()
    try:
        # Отключаем угадывание типов и NA: пустой ответ допустим, ID всегда строка.
        answer = pd.read_csv(
            args.answer, dtype=str, keep_default_na=False, encoding="utf-8"
        )
        # Одного потока достаточно для ID, дочерний CLI не создаёт лишний Arrow pool.
        queries = pd.read_parquet(args.queries, columns=["query_id"], use_threads=False)
        items = pd.read_parquet(args.items, columns=["item_id"], use_threads=False)
        report = validate_submission(answer, queries, items)
    except (ValueError, OSError) as error:
        parser.error(str(error))
    print(json.dumps(report, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/predict.py
"""Получить answer.csv из кандидатов и проверить файл перед отправкой."""

import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoost

from avito_retrieval.evaluation import RESERVED_COLUMNS
from avito_retrieval.submission import validate_submission, write_submission


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_candidates(features, queries, corpus, allowed_items):
    """Проверить позиционные индексы до обращения к модели."""
    if queries.empty:
        raise ValueError("benchmark queries must not be empty")
    empty = pd.DataFrame({"query_id": queries["query_id"], "answer": ""})
    validate_submission(empty, queries, allowed_items)
    allowed_count = len(allowed_items)
    if (
        corpus["item_id"].iloc[:allowed_count].tolist()
        != allowed_items["item_id"].tolist()
    ):
        raise ValueError("corpus prefix does not match the benchmark item order")
    for name in ("qid", "item_idx"):
        if (
            name not in features
            or not pd.api.types.is_integer_dtype(features[name])
            or features[name].isna().any()
        ):
            raise ValueError(f"{name} must contain integer row positions")
    actual = set(features["qid"])
    expected = set(range(len(queries)))
    if actual != expected:
        raise ValueError(
            f"candidate qid coverage mismatch: {len(expected - actual)} missing, "
            f"{len(actual - expected)} extra"
        )
    if features[["qid", "item_idx"]].duplicated().any():
        raise ValueError("duplicate query-item candidate")
    if not features["item_idx"].between(0, allowed_count - 1).all():
        raise ValueError("candidate item_idx outside the benchmark corpus prefix")


def standardize_by_query(scores, qids, query_count):
    """Привести модели к общей шкале только внутри каждого запроса."""
    counts = np.bincount(qids, minlength=query_count)
    means = np.bincount(qids, weights=scores, minlength=query_count) / counts
    centered = scores - means[qids]
    variance = (
        np.bincount(qids, weights=centered * centered, minlength=query_count) / counts
    )
    scale = np.sqrt(variance[qids])
    return np.divide(centered, scale, out=np.zeros_like(centered), where=scale > 0)


def normalize_weights(values, model_count):
    weights = np.asarray(values, dtype=np.float64)
    with np.errstate(over="ignore", invalid="ignore"):
        total = weights.sum()
    if (
        weights.ndim != 1
        or len(weights) != model_count
        or not np.isfinite(weights).all()
        or (weights < 0).any()
        or not np.isfinite(total)
        or total <= 0
    ):
        raise ValueError(
            "weights must match the model count, be finite and non-negative, "
            "and have a positive finite sum"
        )
    return weights / total


def score_models(features, model_paths, weights, query_count, threads):
    qids = features["qid"].to_numpy()
    combined = np.zeros(len(features), dtype=np.float64)
    records = []
    for path, weight in zip(model_paths, weights):
        names_path = path.with_suffix(".feature_names.json")
        names = json.loads(names_path.read_text(encoding="utf-8"))
        if (
            not isinstance(names, list)
            or not names
            or any(not isinstance(name, str) for name in names)
        ):
            raise ValueError(f"invalid feature-name list: {names_path}")
        if len(names) != len(set(names)) or set(names) & RESERVED_COLUMNS:
            raise ValueError(f"duplicate or reserved model feature names: {names_path}")
        missing = set(names) - set(features.columns)
        if missing:
            raise ValueError(f"missing model features: {sorted(missing)}")
        for name in names:
            values = features[name]
            if (
                not pd.api.types.is_numeric_dtype(values)
                or not np.isfinite(values.to_numpy()).all()
            ):
                raise ValueError(f"model feature {name!r} must be numeric and finite")
        model = CatBoost()
        model.load_model(path)
        if model.feature_names_ != names:
            raise ValueError(f"feature-name sidecar does not match model {path}")
        scores = np.asarray(
            model.predict(
                features[names], prediction_type="RawFormulaVal", thread_count=threads
            ),
            dtype=np.float64,
        ).reshape(-1)
        if scores.shape != combined.shape or not np.isfinite(scores).all():
            raise ValueError(f"model {path} returned invalid prediction scores")
        if len(model_paths) > 1:
            scores = standardize_by_query(scores, qids, query_count)
        combined += weight * scores
        records.append(
            {
                "path": str(path),
                "sha256": sha256_file(path),
                "weight": float(weight),
                "feature_names_sha256": sha256_file(names_path),
                "feature_count": len(names),
                "trees": model.tree_count_,
            }
        )
    if not np.isfinite(combined).all():
        raise ValueError("combined prediction scores must be finite")
    return combined, records


def top_predictions(features, scores, item_ids, query_count):
    qids = features["qid"].to_numpy()
    item_indices = features["item_idx"].to_numpy()
    # Порядок кандидатов и разбиение предсказаний на потоки не должны менять ничьи.
    order = np.lexsort((item_indices, -scores, qids))
    sorted_qids = qids[order]
    starts = np.r_[0, np.flatnonzero(np.diff(sorted_qids)) + 1]
    ends = np.r_[starts[1:], len(order)]
    predictions = [[] for _ in range(query_count)]
    for start, end in zip(starts, ends):
        qid = sorted_qids[start]
        selected = item_indices[order[start : min(start + 50, end)]]
        predictions[qid] = item_ids[selected].tolist()
    return predictions


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--features", type=Path, required=True)
    parser.add_argument(
        "--model",
        type=Path,
        action="append",
        required=True,
        help="Repeat this option to combine several models",
    )
    parser.add_argument(
        "--weights",
        type=float,
        nargs="+",
        help="One non-negative weight per model, equal weights by default",
    )
    parser.add_argument("--output", type=Path, default=Path("answer.csv"))
    parser.add_argument(
        "--queries", type=Path, default=Path("data/raw/benchmark_queries.parquet")
    )
    parser.add_argument(
        "--corpus", type=Path, default=Path("data/processed/corpus.parquet")
    )
    parser.add_argument(
        "--allowed-items", type=Path, default=Path("data/raw/benchmark_items.parquet")
    )
    parser.add_argument("--threads", type=int, default=4)
    args = parser.parse_args()
    try:
        weights = normalize_weights(
            args.weights if args.weights is not None else [1.0] * len(args.model),
            len(args.model),
        )
    except ValueError as error:
        parser.error(str(error))
    queries = pd.read_parquet(args.queries, columns=["query_id"], use_threads=False)
    corpus = pd.read_parquet(args.corpus, columns=["item_id"], use_threads=False)
    allowed_items = pd.read_parquet(
        args.allowed_items, columns=["item_id"], use_threads=False
    )
    features = pd.read_parquet(args.features, use_threads=False)
    validate_candidates(features, queries, corpus, allowed_items)
    scores, models = score_models(
        features, args.model, weights, len(queries), args.threads
    )
    predictions = top_predictions(
        features, scores, corpus["item_id"].to_numpy(), len(queries)
    )
    write_submission(queries["query_id"], predictions, args.output, allowed_items)
    # Повторное чтение проверяет сохранённый CSV, а не только данные в памяти.
    restored = pd.read_csv(
        args.output, dtype=str, keep_default_na=False, encoding="utf-8"
    )
    submission = validate_submission(restored, queries, allowed_items)
    counts = np.bincount(features["qid"].to_numpy(), minlength=len(queries))
    report = {
        "submission": submission,
        "output_path": str(args.output),
        "csv_sha256": sha256_file(args.output),
        "features_path": str(args.features),
        "features_sha256": sha256_file(args.features),
        "models": models,
        "combination": "within_query_zscore" if len(models) > 1 else "raw_model_score",
        "queries_sha256": sha256_file(args.queries),
        "allowed_items_sha256": sha256_file(args.allowed_items),
        "candidate_pool": {
            "rows": len(features),
            "min": int(counts.min()),
            "max": int(counts.max()),
            "mean": float(counts.mean()),
            "queries_with_fewer_than_50": int((counts < 50).sum()),
        },
    }
    args.output.with_suffix(".report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/compare_models.py
"""Сравнить модели и их небольшие ансамбли на одном наборе кандидатов."""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from avito_retrieval.evaluation import (
    RESERVED_COLUMNS,
    evaluate_candidates,
    load_split_queries,
    reference_empty_fraction,
)

# Соседний CLI содержит ту же нормализацию, которая применяется при отправке.
if __package__:
    from .predict import score_models, sha256_file, standardize_by_query
else:
    from predict import score_models, sha256_file, standardize_by_query


def paired_bootstrap(current, baseline, fraction, iterations=2000, seed=42):
    """Сравнивать одинаковые запросы, сохраняя парность предсказаний."""
    if baseline["query_id"].duplicated().any():
        raise ValueError("baseline must have exactly one row per query_id")
    reference = baseline.set_index("query_id")
    if not set(current["query_id"]).issubset(reference.index):
        raise ValueError("baseline is missing selected query IDs")
    old = reference.loc[current["query_id"], "recall@50"].to_numpy(dtype=np.float64)
    if not np.isfinite(old).all() or (old < 0).any() or (old > 1).any():
        raise ValueError("baseline Recall@50 must be finite and between zero and one")
    delta = current["recall@50"].to_numpy() - old
    if not len(delta):
        raise ValueError("bootstrap requires at least one query")
    rng = np.random.default_rng(seed)
    samples = rng.integers(0, len(delta), size=(iterations, len(delta)))
    draws = delta[samples].mean(axis=1)
    result = {
        "recall@50": {
            "delta": float(delta.mean()),
            "ci95": np.quantile(draws, [0.025, 0.975]).tolist(),
            "bootstrap_positive_fraction": float((draws > 0).mean()),
        }
    }
    del samples
    # Для переноса на benchmark сохраняем его доли групп и ресэмплируем внутри групп.
    weighted_delta, weighted_draws = 0.0, np.zeros(iterations)
    empty = current["filter_empty"].to_numpy(dtype=bool)
    for mask, weight in ((empty, fraction), (~empty, 1 - fraction)):
        if weight == 0:
            continue
        values = delta[mask]
        if not len(values):
            result["poststratified_recall@50"] = None
            return result
        samples = rng.integers(0, len(values), size=(iterations, len(values)))
        weighted_delta += weight * values.mean()
        weighted_draws += weight * values[samples].mean(axis=1)
    result["poststratified_recall@50"] = {
        "delta": float(weighted_delta),
        "ci95": np.quantile(weighted_draws, [0.025, 0.975]).tolist(),
        "bootstrap_positive_fraction": float((weighted_draws > 0).mean()),
    }
    return result


def required_features(paths):
    result = {"qid", "item_idx"}
    for path in paths:
        names = json.loads(
            path.with_suffix(".feature_names.json").read_text(encoding="utf-8")
        )
        if (
            not isinstance(names, list)
            or not names
            or any(not isinstance(name, str) for name in names)
        ):
            raise ValueError(f"invalid model feature names for {path}")
        if len(names) != len(set(names)) or set(names) & RESERVED_COLUMNS:
            raise ValueError(f"duplicate or reserved model feature names for {path}")
        result.update(names)
    return sorted(result)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--features", type=Path, required=True)
    parser.add_argument("--model", type=Path, action="append", required=True)
    parser.add_argument("--output", type=Path, default=Path("reports/comparison.json"))
    parser.add_argument("--split", choices=["dev", "audit"], default="dev")
    parser.add_argument(
        "--queries", type=Path, default=Path("data/processed/eval_queries.parquet")
    )
    parser.add_argument(
        "--truth", type=Path, default=Path("data/processed/eval_truth.parquet")
    )
    parser.add_argument(
        "--corpus", type=Path, default=Path("data/processed/corpus.parquet")
    )
    parser.add_argument(
        "--reference-queries",
        type=Path,
        default=Path("data/raw/benchmark_queries.parquet"),
    )
    parser.add_argument("--baseline-per-query", type=Path)
    parser.add_argument(
        "--blend-grid",
        type=float,
        nargs="+",
        help="Weights of the first model, two models only, default: 0.25 0.5 0.75",
    )
    parser.add_argument("--no-blends", action="store_true")
    parser.add_argument("--bootstrap", type=int, default=2000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--threads", type=int, default=4)
    args = parser.parse_args()
    if args.bootstrap < 1:
        parser.error("bootstrap must be positive")
    names = [path.stem for path in args.model]
    if len(names) != len(set(names)):
        parser.error("model filenames must have different stems")
    if args.blend_grid is not None and len(args.model) != 2:
        parser.error("blend-grid requires exactly two models")
    grid = args.blend_grid if args.blend_grid is not None else [0.25, 0.5, 0.75]
    if any(not np.isfinite(value) or not 0 < value < 1 for value in grid):
        parser.error("blend-grid weights must be finite and between zero and one")
    queries = load_split_queries(args.queries, [args.split])
    if queries.empty:
        raise ValueError("the selected split contains no queries")
    features = pd.read_parquet(
        args.features,
        columns=required_features(args.model),
        filters=[("qid", "in", queries["qid"].tolist())],
        use_threads=False,
    )
    truth = pd.read_parquet(
        args.truth, filters=[("split", "=", args.split)], use_threads=False
    )
    corpus = pd.read_parquet(args.corpus, columns=["item_id"], use_threads=False)
    fraction = reference_empty_fraction(args.reference_queries)
    candidates = features[["qid", "item_idx"]].copy()
    scores, model_metadata = {}, {}
    for name, path in zip(names, args.model):
        scores[name], records = score_models(
            features, [path], [1.0], len(queries), args.threads
        )
        model_metadata[name] = {"kind": "model", "models": records}
    if len(names) > 1 and not args.no_blends:
        present, dense_qids = np.unique(features["qid"].to_numpy(), return_inverse=True)
        normalized = {
            name: standardize_by_query(scores[name], dense_qids, len(present))
            for name in names
        }
        weights_to_try = (
            [[value, 1 - value] for value in grid]
            if len(names) == 2
            else [[1 / len(names)] * len(names)]
        )
        for weights in weights_to_try:
            blend_name = "blend_" + "_".join(f"{weight:g}" for weight in weights)
            scores[blend_name] = sum(
                weight * normalized[name] for weight, name in zip(weights, names)
            )
            model_metadata[blend_name] = {
                "kind": "within_query_zscore",
                "model_names": names,
                "weights": weights,
            }
    summaries, per_query_tables = {}, {}
    for name, values in scores.items():
        candidates["score"] = values
        summary, per_query = evaluate_candidates(
            candidates,
            queries,
            truth,
            corpus,
            reference_empty_fraction=fraction,
        )
        summaries[name], per_query_tables[name] = summary, per_query
        print(
            json.dumps(
                {
                    "model": name,
                    "recall@50": summary["recall@50"],
                    "poststratified_recall@50": summary["poststratified_recall@50"],
                }
            ),
            flush=True,
        )
    if args.baseline_per_query:
        baseline = pd.read_parquet(args.baseline_per_query, use_threads=False)
        baseline_metadata = {
            "path": str(args.baseline_per_query),
            "sha256": sha256_file(args.baseline_per_query),
        }
    else:
        baseline = per_query_tables[names[0]]
        baseline_metadata = {"model_name": names[0]}
    for name, table in per_query_tables.items():
        summaries[name]["paired_delta_vs_baseline"] = paired_bootstrap(
            table,
            baseline,
            fraction,
            iterations=args.bootstrap,
            seed=args.seed,
        )
    report = {
        "split": args.split,
        "query_count": len(queries),
        "candidate_rows": len(features),
        "features_path": str(args.features),
        "features_sha256": sha256_file(args.features),
        "reference_empty_fraction": fraction,
        "baseline": baseline_metadata,
        "bootstrap": {
            "samples": args.bootstrap,
            "seed": args.seed,
            "interval": "paired percentile 95%, poststratified resampling within filter groups",
        },
        "model_metadata": model_metadata,
        "results": summaries,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    pd.concat(
        [table.assign(model_name=name) for name, table in per_query_tables.items()],
        ignore_index=True,
    ).to_parquet(args.output.with_suffix(".per_query.parquet"), index=False)


if __name__ == "__main__":
    main()


In [ ]:
run_stage("audit")
audit = json.loads((WORK / "reports/audit.json").read_text())
print(json.dumps(audit["results"], ensure_ascii=False, indent=2))


## 11. Рассчитать benchmark и сохранить ответ

Признаки benchmark строятся из исходных таблиц. Оценки моделей стандартизуются отдельно
внутри каждого запроса и усредняются:

$$
z_m(q,d)=\frac{s_m(q,d)-\mu_m(q)}{\sigma_m(q)},\qquad
s(q,d)=0.5z_1(q,d)+0.5z_2(q,d).
$$

При нулевом стандартном отклонении соответствующая оценка равна нулю.
Выбираются 50 объявлений с наибольшими оценками. Проверяются все запросы,
допустимость идентификаторов и отсутствие повторов.

In [ ]:
run_stage("benchmark")
run_stage("predict")
result = json.loads((WORK / "reports/reproducibility.json").read_text())
print("SHA256:", result["answer_sha256"])
print("Совпал с прежним CSV:", result["matches_previous_mps_answer"])


## 12. Скачать результат

Скачайте новый CSV и сохраните отчёты вместе с ним. Именно эта пара соответствует текущему
полному обучению. Архив отчётов не содержит больших признаков или моделей.

In [ ]:
answer = WORK / "answer.csv"
with zipfile.ZipFile(WORK / "run_reports.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted((WORK / "reports").rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(WORK))
if IN_COLAB:
    from google.colab import files

    files.download(str(answer))
    files.download(str(WORK / "run_reports.zip"))
else:
    print("Ответ:", answer)
    print("Отчёты:", WORK / "run_reports.zip")


## Источники

[1] [Версии среды Colab](https://research.google.com/colaboratory/runtime-version-faq.html).

[2] [Ресурсы и ограничения Colab](https://research.google.com/colaboratory/faq.html).

[3] [Установка PyTorch 2.9.1](https://pytorch.org/get-started/previous-versions/#v291).

[4] [Данные конкурса](https://disk.yandex.ru/d/sNhfo0YOjGtufg).

[5] [Исходное решение и проверенные эксперименты](https://github.com/revefffect/avito-contest/blob/main/evidence/INDEX.md).

[6] [E5, фиксированная ревизия](https://huggingface.co/intfloat/multilingual-e5-small/tree/614241f622f53c4eeff9890bdc4f31cfecc418b3).

[7] [uv: установка отдельной версии Python](https://docs.astral.sh/uv/guides/install-python/).